# Phase C/D: GNN model, training and verification

**CV5570 · Team Visionaries · Track B (GNN)**

Phases A and B produced graph shards: FloorPlanCAD (pre-training) and the Indian TOWER drawings (fine-tuning). This notebook builds the model, trains it, and **checks that the training is actually correct** before and after you spend GPU hours.

| Step | What happens | Output |
|---|---|---|
| 0 | Install, config, pull `fpgraph.py` and `config.json` from your dataset repo | same parser and label map as Phases A and B |
| 1 | Write five modules: model and loss, data and augmentation, metrics, training script, checks | `gnnmodel.py`, `gnndata.py`, `gnneval.py`, `train_gnn.py`, `gnncheck.py` |
| 2 | Load the shards from your Hugging Face dataset repo | `phaseC_data/` |
| 3 | **Verification A, before training**: data invariants, augmentation, model sanity, overfit-4-tiles test, learning-rate sweep, go/no-go | a table of PASS / WARN / FAIL with what to change |
| 4 | **D1 dry run**: peak VRAM and time per step, projected total hours | measured numbers |
| 5 | **D2 pre-train** on FloorPlanCAD (single GPU or DDP on all GPUs) | `runs/pretrain/` |
| 6 | **Verification B, after training**: loss, gradient, validation curves, automatic diagnosis, a prediction picture | a table of findings and fixes |
| 7 | **D3 fine-tune** on the Indian shards, then Verification B again | `runs/finetune/` |
| 8 | **D4** sweep the Head 2 threshold on Indian val; final metrics on both test sets | `head2_threshold.json`, `results.json` |
| 9 | Push checkpoints, logs and configs to your Hugging Face output repo | HF |

**Where to start.** Run top to bottom. Fill in the two Hugging Face placeholders in the config cell first (`HF_DATASET_REPO`, `HF_OUTPUT_REPO`). Set `SMOKE_TEST = True` for a tiny synthetic end-to-end check of the whole pipeline (minutes, any runtime, nothing is pushed) before the real run.

**Numbers in this notebook.** The expected-time figures in the plan are *estimates*. Step 4 replaces them with *measured* values from your GPU. The GAT-CADNet score of 85.0 F1 is a *published* reference on the full 35-class-to-12-class FloorPlanCAD setup; ours is trained on a 2,500-drawing subset with a harmonised class set, so treat it as a ceiling to aim at, not a pass mark.

**Design decisions.**
- The model is plain PyTorch (about 300 lines). It does not need PyTorch Geometric, so there is no wheel-matching problem on Colab and `fpgraph.to_pyg` (never tested) is not used.
- Class weights and the edge `pos_weight` are computed from the tiles actually loaded, with the same formula as `fpgraph.compute_stats`, so a limited download still gets matching weights.

## 0. Setup and config
Colab already has `torch`, `numpy`, `scipy`, `pandas` and `matplotlib`. The rest are for `fpgraph` and the Hub.

**Fill in the two placeholders.** Both are Hugging Face repo ids such as `username/name`.
- `HF_DATASET_REPO`: the dataset repo that holds the Phase A shards at its root and the Phase B shards under `indian/` (`graphs/{train,val,test}/*.npz`, plus `fpgraph.py` and `config.json`).
- `HF_OUTPUT_REPO`: where step 9 uploads the trained model and logs. It is created if missing.

The token comes from the Colab secret `HF_TOKEN`. Reading needs read access; step 9 needs **write** access.

In [ ]:
%pip install -q shapely svgelements huggingface_hub pyarrow pandas scipy matplotlib

In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path

SMOKE_TEST = False

HF_DATASET_REPO = "<YOUR_HF_DATASET_REPO>"
HF_OUTPUT_REPO = "<YOUR_HF_OUTPUT_REPO>"
HF_OUTPUT_PRIVATE = True

DATA_DIR = Path("phaseC_data")
RUN_DIR = Path("runs")

MAX_FP_TRAIN_SHARDS = 0
MAX_FP_VAL_SHARDS = 0
MAX_FP_TEST_SHARDS = 2
DOWNLOAD_INDIAN = True

PRETRAIN_EPOCHS = 100
FINETUNE_EPOCHS = 25
LR = 1e-3
FINETUNE_LR = 1e-4
USE_AMP = True
NODE_BUDGET = 12000
MAX_TILES = 8
WANDB_PROJECT = ""
PUSH_LAST_CHECKPOINT = False

if SMOKE_TEST:
    PRETRAIN_EPOCHS, FINETUNE_EPOCHS, USE_AMP, NODE_BUDGET = 3, 3, False, 6000

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")


def require(name, value):
    if value.startswith("<"):
        raise ValueError(f"Set {name} in the config cell before running this step")


print("smoke test:", SMOKE_TEST, "| token found:", bool(HF_TOKEN))

## 1. Use the exact Phase A parser, graph settings and label map
Same rule as Phase B: every tile must have been built identically, so `fpgraph.py` and `config.json` come from the dataset repo, not from a re-declaration here.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

require("HF_DATASET_REPO", HF_DATASET_REPO)
for fname in ("fpgraph.py", "config.json"):
    hf_hub_download(HF_DATASET_REPO, fname, repo_type="dataset", local_dir=".", token=HF_TOKEN)
    print("downloaded", fname)

sys.path.insert(0, str(Path(".").resolve()))
import fpgraph
PHASE_A = json.loads(Path("config.json").read_text())
print("classes:", PHASE_A["targets"])

## 2. The modules
Everything reusable is written to a `.py` file with `%%writefile`, so later phases (inference, the manifest adapter) import it without the notebook. Each file is split into a few cells so the explanation sits next to the code.

### 2a. `gnnmodel.py`: the network and the loss

**The data one tile gives the model.** A tile is a graph. Each node is one CAD primitive (line, arc, circle, ...) with 13 features; each edge links two primitives within 300 mm (or collinear) with 8 relative-geometry features. Tiles are stored with every edge once (`i < j`).

**Trace through one forward pass** (N nodes, E stored edges, D = 128, H = 8 heads, 16 dims per head):

1. `node_in`: `[N,13] -> [N,128]`.
2. The edge list is mirrored to `2E` directed edges. The mirrored copy negates `log_len_ratio`, the one edge feature that flips sign when you swap endpoints (the Phase A helper `to_pyg` does not do this).
3. `rse` (relative-geometry embedding): `[2E,8] -> [2E,64]`.
4. Eight stages. In each stage every node `i` attends over its neighbours `j` and itself: score `= q_i . k_j / sqrt(16) + bias(edge embedding)` per head, softmax over the neighbours of `i`, weighted sum of `v_j`. The edge embedding enters as a per-head bias, so *where* a neighbour sits changes how much it is listened to. Softmax runs in float32 even under fp16 autocast, which is where NaNs come from otherwise.
5. Each stage's attention `[2E,8]` is kept. Averaging the two directions gives one `[E,8]` per stage, so the 8 stages give **64 attention scores per edge**.
6. Head 1: `[N,128] -> [N,12]` class logits.
7. Head 2 (the edge classifier): for each stored edge `(i,j)` concatenate the 64 scores, `u_i + u_j`, `|u_i - u_j|` and the edge embedding, then an MLP to one same-instance logit. Sums and absolute differences make it symmetric in `i, j`.

The last layer of both heads starts near zero, so the initial loss is exactly what chance predicts (`ln 12` for nodes). That is one of the checks in step 3.

**Loss.** Class-weighted cross-entropy on nodes (optional focal term), plus `lambda` times weighted BCE on edges where at least one endpoint is part of a thing instance (`edge_mask`). In fine-tuning, an Indian node labelled `other` that the model calls `elevator` is not penalised, because lifts in the Indian data are drawn without lift blocks and are labelled `other`.

Scatter-softmax over incoming edges and the edge-direction fix.

In [ ]:
%%writefile gnnmodel.py
import math
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F

from fpgraph import TARGET_CLASSES, NODE_FEATURES, EDGE_FEATURES

NUM_CLASSES = len(TARGET_CLASSES)
LEN_RATIO = EDGE_FEATURES.index("log_len_ratio")
OTHER_ID = TARGET_CLASSES.index("other")
ELEVATOR_ID = TARGET_CLASSES.index("elevator")


@dataclass
class ModelConfig:
    node_dim: int = len(NODE_FEATURES)
    edge_dim: int = len(EDGE_FEATURES)
    dim: int = 128
    heads: int = 8
    stages: int = 8
    edge_emb: int = 64
    ffn_mult: int = 2
    dropout: float = 0.0
    num_classes: int = NUM_CLASSES

    def to_dict(self):
        return asdict(self)


def scatter_softmax(scores, index, n):
    h = scores.shape[1]
    idx = index[:, None].expand(-1, h)
    top = torch.full((n, h), float("-inf"), dtype=scores.dtype, device=scores.device)
    top = top.scatter_reduce(0, idx, scores, reduce="amax", include_self=True)
    ex = torch.exp(scores - top[index])
    den = torch.zeros((n, h), dtype=scores.dtype, device=scores.device).index_add_(0, index, ex)
    return ex / (den[index] + 1e-16)


def reverse_edge_attr(edge_attr):
    out = edge_attr.clone()
    out[:, LEN_RATIO] = -out[:, LEN_RATIO]
    return out

One attention stage. Pre-norm, residual stream kept in float32.

In [ ]:
%%writefile -a gnnmodel.py


class SparseGATStage(nn.Module):
    def __init__(self, dim, heads, edge_emb, ffn_mult, dropout):
        super().__init__()
        assert dim % heads == 0
        self.h, self.dh = heads, dim // heads
        self.norm1 = nn.LayerNorm(dim)
        self.qkv = nn.Linear(dim, 3 * dim)
        self.edge_bias = nn.Linear(edge_emb, heads)
        self.self_bias = nn.Parameter(torch.zeros(heads))
        self.out = nn.Linear(dim, dim)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, ffn_mult * dim), nn.GELU(), nn.Linear(ffn_mult * dim, dim))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, src, dst, emb):
        n = x.shape[0]
        q, k, v = self.qkv(self.norm1(x)).view(n, 3, self.h, self.dh).unbind(1)
        loops = torch.arange(n, device=x.device)
        s_all, d_all = torch.cat([src, loops]), torch.cat([dst, loops])
        score = (q[d_all] * k[s_all]).sum(-1) / math.sqrt(self.dh)
        bias = torch.cat([self.edge_bias(emb), self.self_bias.to(score.dtype).expand(n, -1)])
        alpha = scatter_softmax((score + bias).float(), d_all, n)
        msg = v[s_all].float() * alpha[..., None]
        agg = torch.zeros(n, self.h, self.dh, dtype=torch.float32, device=x.device).index_add_(0, d_all, msg)
        x = x + self.drop(self.out(agg.reshape(n, -1))).float()
        x = x + self.drop(self.ffn(self.norm2(x))).float()
        return x, alpha[:src.shape[0]]

The full model: encoders, eight stages, the two heads.

In [ ]:
%%writefile -a gnnmodel.py


class FloorGAT(nn.Module):
    def __init__(self, cfg=None):
        super().__init__()
        self.cfg = cfg or ModelConfig()
        c = self.cfg
        self.node_in = nn.Sequential(nn.Linear(c.node_dim, c.dim), nn.GELU(), nn.Linear(c.dim, c.dim))
        self.rse = nn.Sequential(nn.Linear(c.edge_dim, c.edge_emb), nn.GELU(), nn.Linear(c.edge_emb, c.edge_emb))
        self.stages = nn.ModuleList(
            [SparseGATStage(c.dim, c.heads, c.edge_emb, c.ffn_mult, c.dropout) for _ in range(c.stages)])
        self.final_norm = nn.LayerNorm(c.dim)
        self.head1 = nn.Sequential(nn.Linear(c.dim, c.dim), nn.GELU(), nn.Linear(c.dim, c.num_classes))
        self.node_to_edge = nn.Linear(c.dim, c.edge_emb)
        cee_in = c.stages * c.heads + 3 * c.edge_emb
        self.head2 = nn.Sequential(nn.LayerNorm(cee_in), nn.Linear(cee_in, 128), nn.GELU(),
                                   nn.Linear(128, 64), nn.GELU(), nn.Linear(64, 1))
        for last in (self.head1[-1], self.head2[-1]):
            nn.init.normal_(last.weight, std=0.01)
            nn.init.zeros_(last.bias)

    def forward(self, x, edge_index, edge_attr, return_attn=False):
        e = edge_index.shape[1]
        src = torch.cat([edge_index[0], edge_index[1]])
        dst = torch.cat([edge_index[1], edge_index[0]])
        emb = self.rse(torch.cat([edge_attr, reverse_edge_attr(edge_attr)]))
        h = self.node_in(x).float()
        attn = []
        for stage in self.stages:
            h, a = stage(h, src, dst, emb)
            attn.append((a[:e] + a[e:]) / 2)
        h = self.final_norm(h)
        node_logits = self.head1(h).float()
        u = self.node_to_edge(h).float()
        i, j = edge_index
        sym = (emb[:e].float() + emb[e:].float()) / 2
        feat = torch.cat([torch.cat(attn, 1), u[i] + u[j], (u[i] - u[j]).abs(), sym], 1)
        out = {"node_logits": node_logits, "edge_logits": self.head2(feat).squeeze(-1).float()}
        if return_attn:
            out["attn"] = torch.stack(attn, 1)
        return out


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

The loss.

In [ ]:
%%writefile -a gnnmodel.py


class PanopticLoss(nn.Module):
    def __init__(self, class_weights, edge_pos_weight, lam=2.0, gamma=0.0):
        super().__init__()
        self.register_buffer("cw", torch.as_tensor(class_weights, dtype=torch.float32))
        self.register_buffer("pw", torch.tensor(float(edge_pos_weight)))
        self.lam, self.gamma = lam, gamma

    def forward(self, out, y, edge_y, edge_mask, node_src=None):
        logits = out["node_logits"].float()
        ce = F.cross_entropy(logits, y, reduction="none")
        if self.gamma > 0:
            ce = ce * (1 - torch.exp(-ce)).pow(self.gamma)
        w = self.cw[y]
        if node_src is not None:
            forgive = (node_src == 1) & (y == OTHER_ID) & (logits.argmax(1) == ELEVATOR_ID)
            w = torch.where(forgive, torch.zeros_like(w), w)
        node = (ce * w).sum() / w.sum().clamp(min=1e-6)
        bce = F.binary_cross_entropy_with_logits(out["edge_logits"].float(), edge_y.float(), pos_weight=self.pw,
                                                 reduction="none")
        m = edge_mask.float()
        edge = (bce * m).sum() / m.sum().clamp(min=1.0)
        total = node + self.lam * edge
        return total, {"total": total.detach(), "node": node.detach(), "edge": edge.detach()}

### 2b. `gnndata.py`: loading, augmentation, batching

**Augmentation.** A tile is rotated by a random multiple of 90 degrees about the tile centre and mirrored with probability 0.5, using the stored `pts` (4 control points per primitive, tile-local mm). Only four node features depend on orientation or position: `cos2theta`, `sin2theta`, `cx_tile`, `cy_tile`. They are recomputed with the same formulas as `fpgraph.build_graph`. Every edge feature is invariant (distances, `|sin|` of angle differences, touch, collinear). Labels do not change. Circles and ellipses keep their stored angle features, because their "angle" is an artefact of how their 4 points are numbered. Step 3 proves this: the identity transform must reproduce the stored features exactly.

**Batching.** Tiles are packed up to a node budget (default 12,000 nodes, at most 8 tiles), concatenated with index offsets so one forward pass handles the whole batch. For DDP every rank builds the same batch list from the same seed and takes every `world`-th batch; the list is padded by repeating batches so all ranks take the same number of steps.

Load shards, class weights, and blending of two sources.

In [ ]:
%%writefile gnndata.py
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset

from fpgraph import load_shard, get_tile, NODE_FEATURES, TARGET_CLASSES

TILE_MM = 10000.0
NUM_CLASSES = len(TARGET_CLASSES)
F_COS2T, F_SIN2T = NODE_FEATURES.index("cos2theta"), NODE_FEATURES.index("sin2theta")
F_CX, F_CY = NODE_FEATURES.index("cx_tile"), NODE_FEATURES.index("cy_tile")


def load_tiles(root, split, src, max_shards=None):
    files = sorted((Path(root) / "graphs" / split).glob("*.npz"))
    if max_shards:
        files = files[:max_shards]
    tiles = []
    for f in files:
        shard = load_shard(f)
        for k in range(len(shard["node_ptr"]) - 1):
            t = get_tile(shard, k)
            t["src"] = src
            tiles.append(t)
    return tiles


def class_stats(tiles, num_classes=NUM_CLASSES):
    counts = np.zeros(num_classes, dtype=np.int64)
    pos = masked = 0
    for t in tiles:
        counts += np.bincount(t["y"].astype(np.int64), minlength=num_classes)
        pos += int(t["edge_y"].sum())
        masked += int(t["edge_mask"].sum())
    freq = counts / max(counts.sum(), 1)
    present = counts > 0
    med = np.median(freq[present]) if present.any() else 1.0
    weights = np.where(present, np.clip(med / np.maximum(freq, 1e-12), 0.1, 10.0), 0.0)
    return {"counts": counts, "weights": weights, "pos_weight": float((masked - pos) / max(pos, 1))}


def blend_stats(stats_list):
    w = np.stack([s["weights"] for s in stats_list])
    present = w > 0
    weights = np.where(present.any(0), (w * present).sum(0) / np.maximum(present.sum(0), 1), 0.0)
    return {"counts": sum(s["counts"] for s in stats_list), "weights": weights,
            "pos_weight": float(np.mean([s["pos_weight"] for s in stats_list]))}

Rotation, mirror and feature recomputation.

In [ ]:
%%writefile -a gnndata.py


def transform_pts(pts, k, mirror, tile_size=TILE_MM):
    p = pts.astype(np.float64) - tile_size / 2
    if mirror:
        p[..., 0] = -p[..., 0]
    for _ in range(k % 4):
        p = np.stack([-p[..., 1], p[..., 0]], -1)
    return (p + tile_size / 2).astype(np.float32)


def recompute_node_features(x, pts, ptype, tile_size=TILE_MM):
    out = x.copy()
    p = pts.astype(np.float64)
    d = p[:, 3] - p[:, 0]
    th = np.arctan2(d[:, 1], d[:, 0])
    mid = p.mean(1)
    oriented = ~np.isin(ptype, (2, 3))
    out[oriented, F_COS2T] = np.cos(2 * th)[oriented]
    out[oriented, F_SIN2T] = np.sin(2 * th)[oriented]
    out[:, F_CX] = mid[:, 0] / tile_size
    out[:, F_CY] = mid[:, 1] / tile_size
    return out


def node_length_mm(x):
    return np.expm1(x[:, NODE_FEATURES.index("log_length_m")].astype(np.float64)) * 1000.0

Collate, batch packing, DDP sharding and the dataset.

In [ ]:
%%writefile -a gnndata.py


def collate(tiles, transforms=None):
    xs, eis, eas, ys, insts, eys, ems, bs, srcs = ([] for _ in range(9))
    node_ptr, edge_ptr = [0], [0]
    n_off = inst_off = 0
    for b, t in enumerate(tiles):
        x = t["x"]
        if transforms is not None and (transforms[b][0] % 4 or transforms[b][1]):
            k, mirror = transforms[b]
            x = recompute_node_features(x, transform_pts(t["pts"], k, mirror), t["ptype"])
        n = len(t["y"])
        inst = t["inst"].astype(np.int64)
        xs.append(x)
        eis.append(t["edge_index"].astype(np.int64) + n_off)
        eas.append(t["edge_attr"])
        ys.append(t["y"].astype(np.int64))
        insts.append(np.where(inst >= 0, inst + inst_off, -1))
        eys.append(t["edge_y"].astype(np.float32))
        ems.append(t["edge_mask"].astype(bool))
        bs.append(np.full(n, b, dtype=np.int64))
        srcs.append(np.full(n, t.get("src", 0), dtype=np.int64))
        if n and inst.max() >= 0:
            inst_off += int(inst.max()) + 1
        n_off += n
        node_ptr.append(n_off)
        edge_ptr.append(edge_ptr[-1] + t["edge_index"].shape[1])
    cat = lambda a, ax=0: torch.from_numpy(np.concatenate(a, axis=ax))
    return {
        "x": cat(xs), "edge_index": cat(eis, 1), "edge_attr": cat(eas), "y": cat(ys), "inst": cat(insts),
        "edge_y": cat(eys), "edge_mask": cat(ems), "batch": cat(bs), "node_src": cat(srcs),
        "node_ptr": torch.tensor(node_ptr), "edge_ptr": torch.tensor(edge_ptr),
    }


def to_device(batch, device):
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}


def make_batches(sizes, idx, node_budget, max_tiles, rng=None):
    order = np.asarray(idx)
    if rng is not None:
        order = order[rng.permutation(len(order))]
    batches, cur, n = [], [], 0
    for i in order:
        s = int(sizes[i])
        if cur and (n + s > node_budget or len(cur) >= max_tiles):
            batches.append(cur)
            cur, n = [], 0
        cur.append(int(i))
        n += s
    if cur:
        batches.append(cur)
    return batches


def pad_shard(batches, rank, world):
    if not batches:
        return []
    pad = (-len(batches)) % world
    batches = batches + [batches[i % len(batches)] for i in range(pad)]
    return batches[rank::world]


class BatchDataset(Dataset):
    def __init__(self, tiles, batches, augment=False, seed=0, epoch=0):
        self.tiles, self.batches, self.augment, self.seed, self.epoch = tiles, batches, augment, seed, epoch

    def __len__(self):
        return len(self.batches)

    def __getitem__(self, i):
        group = [self.tiles[j] for j in self.batches[i]]
        transforms = None
        if self.augment:
            rng = np.random.default_rng([self.seed, self.epoch, int(self.batches[i][0])])
            transforms = [(int(rng.integers(4)), bool(rng.integers(2))) for _ in group]
        return collate(group, transforms)

### 2c. `gnneval.py`: metrics and post-processing

**Semantic metrics.** A confusion matrix over nodes gives per-class F1, macro F1 (classes that appear) and weighted F1. `wf1` weights each primitive by its length in mm, which is how FloorPlanCAD results are usually reported.

**Instances (panoptic quality).** Trace for one tile, with Head 2 threshold `t`:
1. Predicted class per node = argmax of Head 1.
2. Keep edges whose Head 2 probability is above `t` and whose two endpoints are both predicted as a *thing* class (doors, window, opening, stairs, elevator).
3. Connected components of those edges are predicted instances. Each takes the length-weighted majority class of its nodes.
4. Ground-truth instances come from `inst`. A prediction matches a ground-truth instance of the same class when length-weighted IoU over primitives is above 0.5 (so at most one match each).
5. Per class: `TP, FP, FN` and the sum of matched IoUs. `SQ = sum IoU / TP`, `RQ = TP / (TP + FP/2 + FN/2)`, `PQ = SQ x RQ`. Counts are summed over the whole dataset before dividing, and averaged over the thing classes present.

Stuff classes (wall, curtain wall, railing) have no instances in FloorPlanCAD and are scored by semantic F1 only. Every statistic is a plain sum, so under DDP the ranks simply `all_reduce` the vectors.

Instance decoding and per-tile panoptic counts.

In [ ]:
%%writefile gnneval.py
import json
from pathlib import Path

import numpy as np
import torch
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from fpgraph import TARGET_CLASSES, STUFF_TARGETS
from gnndata import node_length_mm, to_device

NUM_CLASSES = len(TARGET_CLASSES)
IS_THING = np.array([c not in STUFF_TARGETS and c != "other" for c in TARGET_CLASSES])
DEFAULT_THRESHOLDS = (0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)


def decode_instances(pred, prob, edge_index, length, thr):
    n = len(pred)
    thing = IS_THING[pred]
    keep = (prob > thr) & thing[edge_index[0]] & thing[edge_index[1]]
    ei = edge_index[:, keep]
    graph = coo_matrix((np.ones(ei.shape[1]), (ei[0], ei[1])), shape=(n, n))
    _, comp = connected_components(graph, directed=False)
    seg = np.full(n, -1, dtype=np.int64)
    if not thing.any():
        return seg, np.zeros(0, dtype=np.int64), np.zeros(0)
    _, inv = np.unique(comp[thing], return_inverse=True)
    votes = np.zeros((inv.max() + 1, NUM_CLASSES))
    np.add.at(votes, (inv, pred[thing]), length[thing])
    seg[thing] = inv
    return seg, votes.argmax(1), votes.sum(1)


def panoptic_counts(y, inst, length, seg, seg_cls, seg_len):
    out = np.zeros((NUM_CLASSES, 4))
    has_gt = inst >= 0
    if has_gt.any():
        gt_ids, gi = np.unique(inst[has_gt], return_inverse=True)
        n_gt = len(gt_ids)
        gt_len = np.bincount(gi, weights=length[has_gt], minlength=n_gt)
        gt_cls = np.zeros(n_gt, dtype=np.int64)
        gt_cls[gi] = y[has_gt]
    else:
        n_gt, gt_cls = 0, np.zeros(0, dtype=np.int64)
    m = len(seg_cls)
    tp = np.zeros(NUM_CLASSES)
    iou_sum = np.zeros(NUM_CLASSES)
    both = has_gt & (seg >= 0)
    if both.any():
        full_gi = np.full(len(y), -1, dtype=np.int64)
        full_gi[has_gt] = gi
        key = full_gi[both] * m + seg[both]
        u, inv = np.unique(key, return_inverse=True)
        inter = np.bincount(inv, weights=length[both])
        g_idx, p_idx = u // m, u % m
        iou = inter / (gt_len[g_idx] + seg_len[p_idx] - inter)
        ok = (iou > 0.5) & (gt_cls[g_idx] == seg_cls[p_idx])
        np.add.at(tp, gt_cls[g_idx][ok], 1)
        np.add.at(iou_sum, gt_cls[g_idx][ok], iou[ok])
    n_pred_c = np.bincount(seg_cls, minlength=NUM_CLASSES)
    n_gt_c = np.bincount(gt_cls, minlength=NUM_CLASSES)
    out[:, 0], out[:, 1], out[:, 2], out[:, 3] = tp, n_pred_c - tp, n_gt_c - tp, iou_sum
    return out

The evaluator that accumulates, reduces and summarises.

In [ ]:
%%writefile -a gnneval.py


class Evaluator:
    def __init__(self, thresholds=DEFAULT_THRESHOLDS, num_classes=NUM_CLASSES):
        self.thr, self.C = list(thresholds), num_classes
        T = len(self.thr)
        self.conf = np.zeros((self.C, self.C))
        self.conf_len = np.zeros((self.C, self.C))
        self.pq = np.zeros((T, self.C, 4))
        self.edge = np.zeros((T, 3))
        self.loss = np.zeros(4)

    def update(self, batch, out, parts):
        C = self.C
        pred = out["node_logits"].float().argmax(1).cpu().numpy()
        y = batch["y"].cpu().numpy()
        length = node_length_mm(batch["x"].cpu().numpy())
        inst = batch["inst"].cpu().numpy()
        ei = batch["edge_index"].cpu().numpy()
        prob = torch.sigmoid(out["edge_logits"].float()).cpu().numpy()
        ey = batch["edge_y"].cpu().numpy() > 0.5
        em = batch["edge_mask"].cpu().numpy()
        nptr, eptr = batch["node_ptr"].cpu().numpy(), batch["edge_ptr"].cpu().numpy()
        self.conf += np.bincount(y * C + pred, minlength=C * C).reshape(C, C)
        self.conf_len += np.bincount(y * C + pred, weights=length, minlength=C * C).reshape(C, C)
        for t, thr in enumerate(self.thr):
            p = prob > thr
            self.edge[t] += [(p & ey & em).sum(), (p & ~ey & em).sum(), (~p & ey & em).sum()]
            for b in range(len(nptr) - 1):
                a, z, c, d = nptr[b], nptr[b + 1], eptr[b], eptr[b + 1]
                seg, cls, slen = decode_instances(pred[a:z], prob[c:d], ei[:, c:d] - a, length[a:z], thr)
                self.pq[t] += panoptic_counts(y[a:z], inst[a:z], length[a:z], seg, cls, slen)
        self.loss += [float(parts["total"]), float(parts["node"]), float(parts["edge"]), 1.0]

    def vector(self):
        return np.concatenate([self.conf.ravel(), self.conf_len.ravel(), self.pq.ravel(), self.edge.ravel(), self.loss])

    def load(self, vec):
        C, T = self.C, len(self.thr)
        sizes = [C * C, C * C, T * C * 4, T * 3, 4]
        parts = np.split(np.asarray(vec, dtype=np.float64), np.cumsum(sizes)[:-1])
        self.conf, self.conf_len = parts[0].reshape(C, C), parts[1].reshape(C, C)
        self.pq, self.edge, self.loss = parts[2].reshape(T, C, 4), parts[3].reshape(T, 3), parts[4]

    def summarize(self, thr=0.7):
        t = int(np.argmin(np.abs(np.array(self.thr) - thr)))
        f1c, sup, acc = f1_from_conf(self.conf)
        f1l, supl, _ = f1_from_conf(self.conf_len)
        pq = pq_from_counts(self.pq[t])
        tp, fp, fn = self.edge[t]
        prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        names = TARGET_CLASSES
        n = max(self.loss[3], 1)
        return {
            "thr": self.thr[t], "acc": acc,
            "f1_macro": float(f1c[sup > 0].mean()) if (sup > 0).any() else 0.0,
            "f1_weighted": float((f1c * sup).sum() / max(sup.sum(), 1)),
            "wf1": float((f1l * supl).sum() / max(supl.sum(), 1e-9)),
            "f1_per_class": {c: float(v) for c, v in zip(names, f1c)},
            "support": {c: int(v) for c, v in zip(names, sup)},
            "pq": pq["pq"], "rq": pq["rq"], "sq": pq["sq"], "n_gt_instances": pq["n_gt"],
            "pq_per_class": {c: float(v) for c, v in zip(names, pq["pq_c"])},
            "edge_precision": float(prec), "edge_recall": float(rec),
            "edge_f1": float(2 * prec * rec / max(prec + rec, 1e-9)),
            "loss": float(self.loss[0] / n), "loss_node": float(self.loss[1] / n), "loss_edge": float(self.loss[2] / n),
            "conf": self.conf.tolist(),
            "sweep": [{"thr": th, "pq": pq_from_counts(self.pq[i])["pq"],
                       "edge_f1": float(2 * self.edge[i, 0] / max(2 * self.edge[i, 0] + self.edge[i, 1] + self.edge[i, 2], 1))}
                      for i, th in enumerate(self.thr)],
        }


def f1_from_conf(conf):
    tp = np.diag(conf)
    fp, fn = conf.sum(0) - tp, conf.sum(1) - tp
    den = 2 * tp + fp + fn
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1e-12), 0.0)
    return f1, conf.sum(1), float(tp.sum() / max(conf.sum(), 1e-12))


def pq_from_counts(c):
    tp, fp, fn, iou = c[:, 0], c[:, 1], c[:, 2], c[:, 3]
    den = tp + 0.5 * fp + 0.5 * fn
    pq_c = np.where(den > 0, iou / np.maximum(den, 1e-12), 0.0)
    present = IS_THING & (tp + fn > 0)
    rq_c = np.where(den > 0, tp / np.maximum(den, 1e-12), 0.0)
    sq_c = np.where(tp > 0, iou / np.maximum(tp, 1e-12), 0.0)
    mean = lambda v: float(v[present].mean()) if present.any() else float("nan")
    return {"pq": mean(pq_c), "rq": mean(rq_c), "sq": mean(sq_c), "pq_c": pq_c, "n_gt": int((tp + fn)[IS_THING].sum())}


@torch.no_grad()
def evaluate(model, loader, loss_fn, device, thresholds=DEFAULT_THRESHOLDS, amp=False):
    model.eval()
    ev = Evaluator(thresholds)
    for batch in loader:
        batch = to_device(batch, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            out = model(batch["x"], batch["edge_index"], batch["edge_attr"])
        _, parts = loss_fn(out, batch["y"], batch["edge_y"], batch["edge_mask"], batch["node_src"])
        ev.update(batch, out, parts)
    return ev

Evaluate a saved checkpoint on a split.

In [ ]:
%%writefile -a gnneval.py


def evaluate_checkpoint(ckpt_path, run_config_path, root, split, src, thresholds=DEFAULT_THRESHOLDS, device=None,
                        node_budget=12000, max_tiles=8, amp=False, max_shards=None):
    from torch.utils.data import DataLoader
    from gnnmodel import FloorGAT, ModelConfig, PanopticLoss
    from gnndata import load_tiles, make_batches, BatchDataset
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = FloorGAT(ModelConfig(**ck["cfg"])).to(device)
    model.load_state_dict(ck["model"])
    rc = json.loads(Path(run_config_path).read_text())
    loss_fn = PanopticLoss(rc["class_weights"], rc["edge_pos_weight"]).to(device)
    tiles = load_tiles(root, split, src, max_shards)
    sizes = np.array([len(t["y"]) for t in tiles])
    batches = make_batches(sizes, np.arange(len(tiles)), node_budget, max_tiles)
    loader = DataLoader(BatchDataset(tiles, batches), batch_size=None, shuffle=False)
    ev = evaluate(model, loader, loss_fn, device, thresholds, amp=amp and device.type == "cuda")
    return ev, len(tiles)

### 2d. `train_gnn.py`: the training script
A script rather than notebook code, so `torchrun` can start one process per GPU and a Colab disconnect only costs the current epoch.

- **Optimiser.** Adam, learning rate multiplied by 0.7 every 20 epochs, 200 warm-up steps, gradient clipping at 1.0.
- **Mixed precision.** fp16 autocast with a gradient scaler. Steps whose gradient is not finite are skipped and *counted*, and the count is reported in the diagnosis.
- **DDP.** Gradients are averaged across ranks. Validation is split across ranks and merged by summing the statistics.
- **Resume.** `last.pt` holds model, optimiser, scaler, epoch and early-stopping state. Running the same command again resumes.
- **Early stopping.** On validation PQ (falls back to `wf1` when the validation set has no instances); the best model is saved as `best.pt`.
- **Fine-tune mode.** Loads `--init`, uses all Indian tiles plus FloorPlanCAD tiles making up about 25% of each epoch, blends the two class-weight vectors, and freezes the first stages with a zero learning rate for the first epochs. (A zero learning rate is used instead of `requires_grad=False` because switching `requires_grad` mid-run breaks DDP.)
- **Logs.** `steps.jsonl` (every 10 steps: loss parts, gradient norm, lr, scaler scale) and `metrics.jsonl` (per epoch and per validation). The diagnosis in step 6 reads these.

Arguments.

In [ ]:
%%writefile train_gnn.py
import os
import sys
import json
import time
import argparse
from pathlib import Path

import numpy as np
import torch
import torch.distributed as dist
from torch.utils.data import DataLoader
from torch.nn.parallel import DistributedDataParallel as DDP

from gnnmodel import FloorGAT, ModelConfig, PanopticLoss, count_parameters
from gnndata import load_tiles, class_stats, blend_stats, make_batches, pad_shard, BatchDataset, to_device
from gnneval import Evaluator, evaluate, DEFAULT_THRESHOLDS


def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--fp-dir", required=True)
    p.add_argument("--ind-dir", default="")
    p.add_argument("--out", required=True)
    p.add_argument("--mode", choices=["pretrain", "finetune"], default="pretrain")
    p.add_argument("--epochs", type=int, default=100)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--lr-step", type=int, default=20)
    p.add_argument("--lr-gamma", type=float, default=0.7)
    p.add_argument("--warmup-steps", type=int, default=200)
    p.add_argument("--grad-clip", type=float, default=1.0)
    p.add_argument("--lam", type=float, default=2.0)
    p.add_argument("--focal-gamma", type=float, default=0.0)
    p.add_argument("--thr", type=float, default=0.7)
    p.add_argument("--node-budget", type=int, default=12000)
    p.add_argument("--max-tiles", type=int, default=8)
    p.add_argument("--dim", type=int, default=128)
    p.add_argument("--heads", type=int, default=8)
    p.add_argument("--stages", type=int, default=8)
    p.add_argument("--dropout", type=float, default=0.0)
    p.add_argument("--amp", action="store_true")
    p.add_argument("--no-aug", action="store_true")
    p.add_argument("--eval-every", type=int, default=2)
    p.add_argument("--patience", type=int, default=5)
    p.add_argument("--log-every", type=int, default=10)
    p.add_argument("--val-set", choices=["fp", "ind"], default="fp")
    p.add_argument("--aux-val", choices=["", "fp", "ind"], default="")
    p.add_argument("--max-fp-shards", type=int, default=0)
    p.add_argument("--max-val-tiles", type=int, default=0)
    p.add_argument("--max-train-tiles", type=int, default=0)
    p.add_argument("--max-steps", type=int, default=0)
    p.add_argument("--init", default="")
    p.add_argument("--freeze-stages", type=int, default=0)
    p.add_argument("--freeze-epochs", type=int, default=0)
    p.add_argument("--fp-mix", type=float, default=0.25)
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--wandb", default="")
    p.add_argument("--no-resume", action="store_true")
    return p.parse_args(argv)

Distributed setup, loaders, learning-rate rule and validation.

In [ ]:
%%writefile -a train_gnn.py


def setup_dist():
    world = int(os.environ.get("WORLD_SIZE", "1"))
    rank = int(os.environ.get("RANK", "0"))
    local = int(os.environ.get("LOCAL_RANK", "0"))
    cuda = torch.cuda.is_available()
    if world > 1:
        dist.init_process_group("nccl" if cuda else "gloo")
    if cuda:
        torch.cuda.set_device(local)
    return rank, world, local, torch.device("cuda", local) if cuda else torch.device("cpu")


def reduce_sum(values, device):
    t = torch.tensor(values, dtype=torch.float64, device=device)
    if dist.is_available() and dist.is_initialized():
        dist.all_reduce(t)
    return t.cpu().numpy()


def reduce_max(value, device):
    t = torch.tensor([value], dtype=torch.float64, device=device)
    if dist.is_available() and dist.is_initialized():
        dist.all_reduce(t, op=dist.ReduceOp.MAX)
    return float(t.item())


def make_loader(tiles, batches, args, augment, epoch):
    ds = BatchDataset(tiles, batches, augment=augment, seed=args.seed, epoch=epoch)
    w = args.workers
    return DataLoader(ds, batch_size=None, shuffle=False, num_workers=w, prefetch_factor=2 if w > 0 else None)


def subsample(tiles, n, seed):
    if n and len(tiles) > n:
        keep = np.random.default_rng(seed).choice(len(tiles), n, replace=False)
        return [tiles[i] for i in sorted(keep)]
    return tiles


def append_jsonl(path, row):
    with open(path, "a") as f:
        f.write(json.dumps(row) + "\n")


def lr_for(group, epoch, step, args):
    if epoch < group["frozen_until"]:
        return 0.0
    base = group["base_lr"] * args.lr_gamma ** (epoch // args.lr_step)
    return base * min(1.0, (step + 1) / max(args.warmup_steps, 1))


def run_validation(model, tiles, loss_fn, args, rank, world, device):
    sizes = np.array([len(t["y"]) for t in tiles])
    batches = make_batches(sizes, np.arange(len(tiles)), args.node_budget, args.max_tiles)[rank::world]
    loader = make_loader(tiles, batches, args, False, 0)
    ev = evaluate(model, loader, loss_fn, device, DEFAULT_THRESHOLDS, amp=args.amp and device.type == "cuda")
    total = Evaluator(DEFAULT_THRESHOLDS)
    total.load(reduce_sum(ev.vector(), device))
    return total.summarize(args.thr)


def select_score(summary):
    return summary["pq"] if summary["n_gt_instances"] > 0 and summary["pq"] == summary["pq"] else summary["wf1"]

The training loop.

In [ ]:
%%writefile -a train_gnn.py


def main(argv=None):
    args = parse_args(argv)
    rank, world, local, device = setup_dist()
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    amp = args.amp and device.type == "cuda"
    finetune = args.mode == "finetune"

    fp_train = load_tiles(args.fp_dir, "train", 0, args.max_fp_shards or None)
    ind_train = load_tiles(args.ind_dir, "train", 1) if finetune else []
    if finetune:
        pool = ind_train + fp_train
    else:
        pool = fp_train
    pool = subsample(pool, args.max_train_tiles, args.seed)
    n_ind = sum(1 for t in pool if t["src"] == 1)
    val_dirs = {"fp": (args.fp_dir, 0), "ind": (args.ind_dir, 1)}
    val_sets = {}
    for name in [args.val_set] + ([args.aux_val] if args.aux_val and args.aux_val != args.val_set else []):
        d, s = val_dirs[name]
        shards = (args.max_fp_shards or None) if name == "fp" else None
        val_sets[name] = subsample(load_tiles(d, "val", s, shards), args.max_val_tiles, args.seed)
    if finetune and n_ind:
        stats = blend_stats([class_stats([t for t in pool if t["src"] == 1]),
                             class_stats([t for t in pool if t["src"] == 0])])
    else:
        stats = class_stats(pool)
    sizes = np.array([len(t["y"]) for t in pool])
    ind_idx = np.array([i for i, t in enumerate(pool) if t["src"] == 1], dtype=int)
    fp_idx = np.array([i for i, t in enumerate(pool) if t["src"] == 0], dtype=int)

    cfg = ModelConfig(dim=args.dim, heads=args.heads, stages=args.stages, dropout=args.dropout)
    torch.manual_seed(args.seed)
    model = FloorGAT(cfg).to(device)
    if args.init:
        ck = torch.load(args.init, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"])
    loss_fn = PanopticLoss(stats["weights"], stats["pos_weight"], args.lam, args.focal_gamma).to(device)

    frozen = [p for s in model.stages[:args.freeze_stages] for p in s.parameters()]
    frozen_ids = {id(p) for p in frozen}
    rest = [p for p in model.parameters() if id(p) not in frozen_ids]
    groups = [{"params": rest, "base_lr": args.lr, "frozen_until": 0, "lr": args.lr}]
    if frozen:
        groups.append({"params": frozen, "base_lr": args.lr, "frozen_until": args.freeze_epochs, "lr": args.lr})
    opt = torch.optim.Adam(groups)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)

    start_epoch, step, best, bad_evals = 0, 0, -1.0, 0
    last_path = out / "last.pt"
    if last_path.exists() and not args.no_resume:
        ck = torch.load(last_path, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        start_epoch, step, best, bad_evals = ck["epoch"] + 1, ck["step"], ck["best"], ck["bad_evals"]
        if rank == 0:
            print(f"resumed from epoch {ck['epoch']}")
    train_model = DDP(model, device_ids=[local] if device.type == "cuda" else None) if world > 1 else model

    if rank == 0:
        run_cfg = {"args": vars(args), "model": cfg.to_dict(), "params": count_parameters(model), "world": world,
                   "device": torch.cuda.get_device_name(local) if device.type == "cuda" else "cpu",
                   "torch": torch.__version__, "train_tiles": len(pool), "indian_tiles": n_ind,
                   "val_tiles": {k: len(v) for k, v in val_sets.items()},
                   "class_weights": stats["weights"].tolist(), "edge_pos_weight": stats["pos_weight"],
                   "class_counts": stats["counts"].tolist()}
        (out / "run_config.json").write_text(json.dumps(run_cfg, indent=1))
        print(json.dumps({k: v for k, v in run_cfg.items() if k != "args"}))
    wb = None
    if rank == 0 and args.wandb:
        import wandb
        wb = wandb.init(project=args.wandb, name=out.name, config=vars(args), resume="allow")

    stop = False
    for epoch in range(start_epoch, args.epochs):
        rng = np.random.default_rng([args.seed, epoch])
        if finetune and len(ind_idx) and len(fp_idx):
            n_fp = int(round(len(ind_idx) * args.fp_mix / (1 - args.fp_mix)))
            idx = np.concatenate([ind_idx, rng.choice(fp_idx, min(n_fp, len(fp_idx)), replace=False)])
        else:
            idx = np.arange(len(pool))
        batches = pad_shard(make_batches(sizes, idx, args.node_budget, args.max_tiles, rng), rank, world)
        loader = make_loader(pool, batches, args, not args.no_aug, epoch)
        for g in opt.param_groups:
            g["lr"] = lr_for(g, epoch, step, args)
        model.train()
        t0 = time.time()
        acc = np.zeros(8)
        gn_max = 0.0
        for batch in loader:
            for g in opt.param_groups:
                g["lr"] = lr_for(g, epoch, step, args)
            batch = to_device(batch, device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                out_t = train_model(batch["x"], batch["edge_index"], batch["edge_attr"])
            total, parts = loss_fn(out_t, batch["y"], batch["edge_y"], batch["edge_mask"], batch["node_src"])
            scaler.scale(total).backward()
            scaler.unscale_(opt)
            gn = float(torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip))
            finite = np.isfinite(gn)
            if amp:
                scaler.step(opt)
                scaler.update()
            elif finite:
                opt.step()
            step += 1
            tl, nl, el = float(parts["total"]), float(parts["node"]), float(parts["edge"])
            acc += [tl if np.isfinite(tl) else 0.0, nl if np.isfinite(nl) else 0.0, el if np.isfinite(el) else 0.0, 1,
                    0 if finite else 1, gn if finite else 0.0, batch["x"].shape[0], len(batch["node_ptr"]) - 1]
            gn_max = max(gn_max, gn if finite else 0.0)
            if rank == 0 and step % args.log_every == 0:
                append_jsonl(out / "steps.jsonl", {"step": step, "epoch": epoch, "loss": tl, "node": nl, "edge": el,
                                                   "grad_norm": gn, "lr": opt.param_groups[0]["lr"],
                                                   "scale": float(scaler.get_scale()) if amp else 1.0})
            if args.max_steps and step >= args.max_steps:
                stop = True
                break
        tot = reduce_sum(acc.tolist(), device)
        seconds = time.time() - t0
        n_steps = max(tot[3], 1)
        row = {"event": "epoch", "epoch": epoch, "step": step, "loss": tot[0] / n_steps, "loss_node": tot[1] / n_steps,
               "loss_edge": tot[2] / n_steps, "steps": int(tot[3]), "skipped": int(tot[4]),
               "grad_mean": tot[5] / max(tot[3] - tot[4], 1), "grad_max": reduce_max(gn_max, device),
               "nodes": int(tot[6]), "tiles": int(tot[7]), "lr": opt.param_groups[0]["lr"], "seconds": seconds,
               "tiles_per_s": tot[7] / max(seconds, 1e-9),
               "gpu_peak_gb": torch.cuda.max_memory_allocated(device) / 2 ** 30 if device.type == "cuda" else 0.0}
        if rank == 0:
            append_jsonl(out / "metrics.jsonl", row)
            print(f"epoch {epoch} loss {row['loss']:.4f} node {row['loss_node']:.4f} edge {row['loss_edge']:.4f} "
                  f"gn {row['grad_mean']:.2f} {seconds:.0f}s")
            if wb:
                wb.log({f"train/{k}": v for k, v in row.items() if isinstance(v, (int, float))}, step=epoch)
        if (epoch + 1) % args.eval_every == 0 or epoch + 1 == args.epochs or stop:
            for name, tiles in val_sets.items():
                s = run_validation(model, tiles, loss_fn, args, rank, world, device)
                if rank == 0:
                    append_jsonl(out / "metrics.jsonl", {"event": "val", "set": name, "epoch": epoch, "summary": s})
                    print(f"  val[{name}] wF1 {s['wf1']:.4f} F1 {s['f1_macro']:.4f} PQ {s['pq']:.4f} "
                          f"edgeF1 {s['edge_f1']:.4f}")
                    if wb:
                        wb.log({f"val_{name}/{k}": v for k, v in s.items() if isinstance(v, float)}, step=epoch)
                if name == args.val_set:
                    score = select_score(s)
                    if score > best:
                        best, bad_evals = score, 0
                        if rank == 0:
                            torch.save({"model": model.state_dict(), "cfg": cfg.to_dict(), "epoch": epoch,
                                        "score": score, "thr": args.thr, "summary": s}, out / "best.pt")
                    else:
                        bad_evals += 1
            if bad_evals >= args.patience:
                stop = True
        if rank == 0:
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "epoch": epoch, "step": step, "best": best, "bad_evals": bad_evals,
                        "cfg": cfg.to_dict()}, last_path)
        if world > 1:
            dist.barrier()
        if stop:
            if rank == 0:
                print("stopping", "(early stop)" if bad_evals >= args.patience else "(max steps)")
            break
    if world > 1:
        dist.destroy_process_group()


if __name__ == "__main__":
    main()

### 2e. `gnncheck.py`: the verification toolkit
These are the functions behind steps 3, 4 and 6. Each check returns a row `status / check / finding / what to change`, so a wrong setup shows up as a named FAIL with an action rather than as a vague bad curve at epoch 40.

| Check | What a failure means |
|---|---|
| tile invariants | broken shards; stop |
| identity augmentation reproduces stored features | recomputed features disagree with Phase A, so augmentation would add noise |
| rotation and mirror are rigid motions, edge geometry preserved | `edge_attr` would no longer match `pts` |
| initial loss about `ln 12` for nodes, matches label mix for edges | head init or class weights are off |
| node-order equivariance | message passing has an indexing bug |
| tiles in one batch do not leak | batch offsets are wrong |
| every parameter gets a finite gradient | dead layer or NaN |
| **overfit 4 tiles** | a correct model memorises 4 tiles in a few hundred steps; if it cannot, the bug is in labels, features or loss, not in tuning |
| learning-rate sweep | divergence or a clearly better learning rate |

Helpers.

In [ ]:
%%writefile gnncheck.py
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

import fpgraph
from fpgraph import TARGET_CLASSES
from gnnmodel import FloorGAT, PanopticLoss, count_parameters
from gnndata import (collate, to_device, transform_pts, recompute_node_features, class_stats, make_batches,
                     node_length_mm, NUM_CLASSES, F_COS2T, F_SIN2T, F_CX, F_CY)

PASS, WARN, FAIL, INFO = "PASS", "WARN", "FAIL", "INFO"
SERIES = ["#2a6fdb", "#e07b00", "#1a9850", "#8e44ad", "#c0392b"]


def row(status, check, finding, action=""):
    return {"status": status, "check": check, "finding": finding, "what to change": action}


def show(rows):
    df = pd.DataFrame(rows)
    try:
        from IPython.display import display
        with pd.option_context("display.max_colwidth", 200, "display.width", 250):
            display(df.style.hide(axis="index").map(
                lambda v: {"PASS": "color:#1a7f37;font-weight:600", "WARN": "color:#b35900;font-weight:600",
                           "FAIL": "color:#c0392b;font-weight:600"}.get(v, ""), subset=["status"]))
    except Exception:
        print(df.to_string(index=False))
    return df


def read_jsonl(path):
    p = Path(path)
    return [json.loads(line) for line in p.read_text().splitlines() if line.strip()] if p.exists() else []

Data, augmentation and model checks.

In [ ]:
%%writefile -a gnncheck.py


def check_data(tiles, name="train"):
    cfg = fpgraph.GraphConfig()
    problems = [(i, p) for i, t in enumerate(tiles) for p in fpgraph.validate_tile(t, cfg)]
    rows = [row(PASS if not problems else FAIL, f"{name}: tile invariants",
                f"{len(tiles)} tiles, {len(problems)} problems" + (f", first: {problems[0]}" if problems else ""),
                "" if not problems else "rebuild shards with fpgraph; do not train on invalid tiles")]
    stats = class_stats(tiles)
    counts = stats["counts"]
    share = counts / max(counts.sum(), 1)
    absent = [c for c, n in zip(TARGET_CLASSES, counts) if n == 0]
    rows.append(row(INFO, f"{name}: class balance", ", ".join(f"{c} {s:.1%}" for c, s in zip(TARGET_CLASSES, share) if s > 0),
                    ""))
    rows.append(row(INFO if absent else PASS, f"{name}: absent classes", ", ".join(absent) or "none",
                    "absent classes get weight 0 and can never be learned from this set" if absent else ""))
    n_inst = np.array([int(t["inst"].max() + 1) if len(t["inst"]) else 0 for t in tiles])
    rows.append(row(PASS if (n_inst > 0).mean() > 0.5 else WARN, f"{name}: tiles with instances",
                    f"{(n_inst > 0).mean():.0%} of tiles have at least one thing instance",
                    "Head 2 needs positives; check the instance-id parsing" if (n_inst > 0).mean() <= 0.5 else ""))
    pos = sum(int(t["edge_y"].sum()) for t in tiles)
    masked = sum(int(t["edge_mask"].sum()) for t in tiles)
    rows.append(row(INFO, f"{name}: Head 2 edges", f"{pos:,} positive of {masked:,} masked edges, pos_weight {stats['pos_weight']:.2f}", ""))
    return rows


def check_augmentation(tiles, n=8):
    rows = []
    sel = tiles[:n]
    ident = max(float(np.abs(recompute_node_features(t["x"], t["pts"], t["ptype"]) - t["x"]).max()) for t in sel)
    rows.append(row(PASS if ident < 1e-3 else FAIL, "augmentation: identity reproduces Phase A features",
                    f"max abs difference {ident:.2e}",
                    "" if ident < 1e-3 else "recompute_node_features disagrees with fpgraph.build_graph; augmentation would inject label-free noise"))
    cyc = max(float(np.abs(transform_pts(transform_pts(transform_pts(transform_pts(t["pts"], 1, False), 1, False), 1, False), 1, False) - t["pts"]).max()) for t in sel)
    mir = max(float(np.abs(transform_pts(transform_pts(t["pts"], 0, True), 0, True) - t["pts"]).max()) for t in sel)
    rows.append(row(PASS if max(cyc, mir) < 0.05 else FAIL, "augmentation: 4 rotations and 2 mirrors return to start",
                    f"rotation drift {cyc:.3g} mm, mirror drift {mir:.3g} mm", "" if max(cyc, mir) < 0.05 else "transform_pts is not a rigid motion"))
    worst = 0.0
    for t in sel:
        if t["edge_index"].shape[1] == 0:
            continue
        i, j = t["edge_index"]
        for k, m in ((1, False), (2, True), (3, True)):
            p0, p1 = t["pts"].astype(np.float64).mean(1), transform_pts(t["pts"], k, m).astype(np.float64).mean(1)
            d0 = np.linalg.norm(p0[i] - p0[j], axis=1)
            d1 = np.linalg.norm(p1[i] - p1[j], axis=1)
            worst = max(worst, float(np.abs(d0 - d1).max()))
    rows.append(row(PASS if worst < 0.05 else FAIL, "augmentation: edge geometry preserved",
                    f"max change in neighbour distance {worst:.3g} mm",
                    "" if worst < 0.05 else "edge_attr would no longer match pts after augmentation"))
    batch = collate(sel, [(1, True)] * len(sel))
    plain = collate(sel)
    same = bool((batch["y"] == plain["y"]).all() and (batch["edge_y"] == plain["edge_y"]).all())
    cx = batch["x"][:, F_CX]
    rows.append(row(PASS if same else FAIL, "augmentation: labels untouched", f"node and edge labels identical: {same}", ""))
    span = (float(cx.min()), float(cx.max()))
    rows.append(row(PASS if -0.6 < span[0] and span[1] < 1.8 else WARN, "augmentation: positions stay in tile range",
                    f"cx range after rotate+mirror {span[0]:.2f} to {span[1]:.2f}", "positions drifting outside the tile mean the rotation centre is wrong"))
    return rows


def _batch(tiles, device):
    return to_device(collate(tiles), device)


def check_model(cfg, tiles, loss_fn, device, seed=0):
    torch.manual_seed(seed)
    model = FloorGAT(cfg).to(device)
    sel = [t for t in tiles if t["edge_index"].shape[1] > 0][:2]
    rows = [row(INFO, "model: parameters", f"{count_parameters(model):,} trainable parameters", "")]
    model.eval()
    b = _batch(sel, device)
    with torch.no_grad():
        out = model(b["x"], b["edge_index"], b["edge_attr"])
        _, parts = loss_fn(out, b["y"], b["edge_y"], b["edge_mask"], b["node_src"])
    finite = bool(torch.isfinite(out["node_logits"]).all() and torch.isfinite(out["edge_logits"]).all())
    rows.append(row(PASS if finite else FAIL, "model: forward is finite", f"shapes {tuple(out['node_logits'].shape)}, {tuple(out['edge_logits'].shape)}",
                    "" if finite else "NaN at initialisation: inspect scatter_softmax and feature scaling"))
    exp_node = float(np.log(cfg.num_classes))
    got_node = float(parts["node"])
    rows.append(row(PASS if abs(got_node - exp_node) / exp_node < 0.25 else WARN, "model: initial node loss is about ln(12)",
                    f"expected {exp_node:.3f}, got {got_node:.3f}",
                    "" if abs(got_node - exp_node) / exp_node < 0.25 else "head initialisation too large or class weights extreme"))
    em, ey = b["edge_mask"].float(), b["edge_y"]
    pos = float((ey * em).sum() / em.sum().clamp(min=1))
    exp_edge = float(np.log(2) * (float(loss_fn.pw) * pos + (1 - pos)))
    got_edge = float(parts["edge"])
    rows.append(row(PASS if abs(got_edge - exp_edge) / max(exp_edge, 1e-9) < 0.25 else WARN, "model: initial edge loss matches the label mix",
                    f"expected {exp_edge:.3f}, got {got_edge:.3f}", "check pos_weight and edge_mask"))
    one = _batch(sel[:1], device)
    n = one["x"].shape[0]
    perm = torch.randperm(n, device=device)
    inv = torch.empty_like(perm)
    inv[perm] = torch.arange(n, device=device)
    with torch.no_grad():
        o1 = model(one["x"], one["edge_index"], one["edge_attr"])
        o2 = model(one["x"][perm], inv[one["edge_index"]], one["edge_attr"])
        dn = float((o1["node_logits"][perm] - o2["node_logits"]).abs().max())
        de = float((o1["edge_logits"] - o2["edge_logits"]).abs().max())
        both = _batch(sel, device)
        ob = model(both["x"], both["edge_index"], both["edge_attr"])
        db = float((ob["node_logits"][:n] - o1["node_logits"]).abs().max())
    rows.append(row(PASS if max(dn, de) < 1e-3 else FAIL, "model: node-order equivariance", f"max difference node {dn:.2e}, edge {de:.2e}",
                    "" if max(dn, de) < 1e-3 else "the model depends on node order; message passing has an indexing bug"))
    rows.append(row(PASS if db < 1e-3 else FAIL, "model: tiles in a batch do not leak into each other", f"max difference {db:.2e}",
                    "" if db < 1e-3 else "batch offsets are wrong in collate or the layer mixes graphs"))
    model.train()
    b = _batch(sel, device)
    model.zero_grad()
    out = model(b["x"], b["edge_index"], b["edge_attr"])
    loss, _ = loss_fn(out, b["y"], b["edge_y"], b["edge_mask"], b["node_src"])
    loss.backward()
    dead = [nm for nm, p in model.named_parameters() if p.grad is None or float(p.grad.abs().max()) == 0.0]
    bad = [nm for nm, p in model.named_parameters() if p.grad is not None and not bool(torch.isfinite(p.grad).all())]
    rows.append(row(PASS if not dead else WARN, "model: every parameter receives gradient",
                    f"{len(dead)} without gradient" + (f": {dead[:4]}" if dead else ""),
                    "" if not dead else "unused layers; check head wiring (zero grads at step 0 can be normal for the zero-init head only)"))
    rows.append(row(PASS if not bad else FAIL, "model: gradients are finite", f"{len(bad)} parameters with NaN or inf", ""))
    return rows

Overfit test, learning-rate sweep and the D1 dry run.

In [ ]:
%%writefile -a gnncheck.py


def _pick(tiles, n, max_nodes=2500):
    cand = [t for t in tiles if 0 < len(t["y"]) <= max_nodes and t["edge_index"].shape[1] > 0]
    cand.sort(key=lambda t: -int(t["inst"].max() + 1 if len(t["inst"]) else 0))
    return cand[:n]


def overfit_test(cfg, tiles, loss_fn, device, steps=300, lr=2e-3, n_tiles=4, seed=0):
    torch.manual_seed(seed)
    model = FloorGAT(cfg).to(device)
    b = _batch(_pick(tiles, n_tiles), device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = []
    for s in range(1, steps + 1):
        model.train()
        opt.zero_grad()
        out = model(b["x"], b["edge_index"], b["edge_attr"])
        loss, parts = loss_fn(out, b["y"], b["edge_y"], b["edge_mask"], b["node_src"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if s % 10 == 0 or s == 1:
            with torch.no_grad():
                acc = float((out["node_logits"].argmax(1) == b["y"]).float().mean())
                p = out["edge_logits"] > 0
                ey, em = b["edge_y"] > 0.5, b["edge_mask"]
                tp, fp, fn = [float(v.sum()) for v in (p & ey & em, p & ~ey & em, ~p & ey & em)]
            hist.append({"step": s, "loss": float(loss), "node_acc": acc, "edge_f1": 2 * tp / max(2 * tp + fp + fn, 1)})
    h = pd.DataFrame(hist)
    last = h.iloc[-1]
    drop = 1 - last["loss"] / h.iloc[0]["loss"]
    rows = [
        row(PASS if last["node_acc"] >= 0.97 else (WARN if last["node_acc"] >= 0.9 else FAIL), "overfit 4 tiles: node accuracy",
            f"{last['node_acc']:.3f} after {steps} steps (loss down {drop:.0%})",
            "" if last["node_acc"] >= 0.97 else "a correct model memorises 4 tiles; try more steps, then suspect a bug in labels, features or the loss"),
        row(PASS if last["edge_f1"] >= 0.8 else (WARN if last["edge_f1"] >= 0.5 else FAIL), "overfit 4 tiles: Head 2 edge F1",
            f"{last['edge_f1']:.3f}", "" if last["edge_f1"] >= 0.8 else "Head 2 is not learning: check edge_mask, edge_y and pos_weight"),
    ]
    return h, rows


def lr_sweep(cfg, tiles, loss_fn, device, lrs=(1e-4, 3e-4, 1e-3, 3e-3), steps=80, seed=0):
    sel = _pick(tiles, 6)
    batches = [_batch(sel[:3], device), _batch(sel[3:6] or sel[:3], device)]
    res = []
    for lr in lrs:
        torch.manual_seed(seed)
        model = FloorGAT(cfg).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        losses = []
        for s in range(steps):
            b = batches[s % 2]
            opt.zero_grad()
            out = model(b["x"], b["edge_index"], b["edge_attr"])
            loss, _ = loss_fn(out, b["y"], b["edge_y"], b["edge_mask"], b["node_src"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(float(loss))
        res.append({"lr": lr, "start": losses[0], "end": float(np.mean(losses[-10:])), "finite": bool(np.isfinite(losses).all())})
    df = pd.DataFrame(res)
    ok = df[df["finite"]]
    best = float(ok.loc[ok["end"].idxmin(), "lr"]) if len(ok) else float("nan")
    return df, best


def dry_run(tiles, cfg, loss_fn, device, budgets=(6000, 12000, 20000), max_tiles=8, steps=4, amp=False, seed=0):
    rng = np.random.default_rng(seed)
    sizes = np.array([len(t["y"]) for t in tiles])
    cases = [("largest single tile", [int(sizes.argmax())])]
    for bud in budgets:
        bs = make_batches(sizes, np.arange(len(tiles)), bud, max_tiles, rng)
        cases.append((f"budget {bud}", max(bs, key=lambda g: sizes[g].sum())))
    res = []
    for name, group in cases:
        torch.manual_seed(seed)
        model = FloorGAT(cfg).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=1e-4)
        scaler = torch.amp.GradScaler("cuda", enabled=amp and device.type == "cuda")
        b = _batch([tiles[i] for i in group], device)
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(device)
        times = []
        for s in range(steps + 1):
            if device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.time()
            opt.zero_grad()
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp and device.type == "cuda"):
                out = model(b["x"], b["edge_index"], b["edge_attr"])
            loss, _ = loss_fn(out, b["y"], b["edge_y"], b["edge_mask"], b["node_src"])
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            if device.type == "cuda":
                torch.cuda.synchronize()
            if s > 0:
                times.append(time.time() - t0)
        sec = float(np.mean(times))
        res.append({"case": name, "tiles": len(group), "nodes": int(b["x"].shape[0]), "edges": int(b["edge_index"].shape[1]),
                    "peak_gb": torch.cuda.max_memory_allocated(device) / 2 ** 30 if device.type == "cuda" else float("nan"),
                    "sec_per_step": sec, "tiles_per_s": len(group) / sec})
    return pd.DataFrame(res)


def project_time(df, case, n_tiles, epochs, world=1):
    r = df[df["case"] == case].iloc[0]
    per_epoch_min = n_tiles / r["tiles_per_s"] / world / 60
    return {"case": case, "measured_tiles_per_s_per_gpu": float(r["tiles_per_s"]), "projected_min_per_epoch": per_epoch_min,
            "projected_hours_total": per_epoch_min * epochs / 60}

Post-training diagnosis, plots and the prediction picture.

In [ ]:
%%writefile -a gnncheck.py


def analyze_run(run_dir, reference_wf1=0.85):
    run = Path(run_dir)
    cfg = json.loads((run / "run_config.json").read_text())
    args = cfg["args"]
    log, steps = read_jsonl(run / "metrics.jsonl"), read_jsonl(run / "steps.jsonl")
    ep = [r for r in log if r["event"] == "epoch"]
    val = [r for r in log if r["event"] == "val" and r["set"] == args["val_set"]]
    if not ep:
        return show([row(FAIL, "log", "no epoch finished", "read the training output for the traceback")]), None
    rows = []
    loss_ep = np.array([r["loss"] for r in ep])
    skipped, tot_steps = sum(r["skipped"] for r in ep), sum(r["steps"] for r in ep)
    bad = (not np.isfinite(loss_ep).all()) or skipped / max(tot_steps, 1) > 0.005
    rows.append(row(FAIL if not np.isfinite(loss_ep).all() else (WARN if bad else PASS), "numerics",
                    f"{skipped} of {tot_steps} steps skipped for non-finite gradients",
                    "lower --lr, keep --grad-clip 1, or drop --amp" if bad else ""))
    sl = np.array([s["loss"] for s in steps if np.isfinite(s["loss"])])
    if len(sl) >= 40:
        k = max(len(sl) // 5, 10)
        drop = 1 - sl[-k:].mean() / sl[:k].mean()
        rows.append(row(PASS if drop >= 0.3 else (WARN if drop >= 0.1 else FAIL), "learning",
                        f"train loss fell {drop:.0%} from the first to the last fifth of logged steps",
                        "" if drop >= 0.3 else "barely learning: run the overfit test, then try lr 3e-4 or 3e-3, and check class weights"))
    elif len(ep) >= 2:
        drop = 1 - loss_ep[-1] / loss_ep[0]
        rows.append(row(PASS if drop >= 0.2 else WARN, "learning", f"epoch loss fell {drop:.0%}", "" if drop >= 0.2 else "train longer or raise lr"))
    last = ep[-1]
    rows.append(row(INFO, "loss composition", f"node {last['loss_node']:.3f}, edge {last['loss_edge']:.3f} x lambda {args['lam']}",
                    "if the edge term dominates and node F1 lags, lower --lam to 1" if args["lam"] * last["loss_edge"] > 3 * last["loss_node"] else ""))
    ratio = np.median([r["grad_max"] / max(r["grad_mean"], 1e-9) for r in ep])
    rows.append(row(WARN if ratio > 10 else PASS, "gradient spikes", f"median max/mean gradient norm {ratio:.1f}",
                    "lower --lr or --grad-clip" if ratio > 10 else ""))
    clipped = np.mean([r["grad_mean"] > 3 * args["grad_clip"] for r in ep])
    rows.append(row(WARN if clipped > 0.8 else PASS, "clipping", f"mean gradient norm above 3x clip in {clipped:.0%} of epochs",
                    "clip is active almost always: lower --lr" if clipped > 0.8 else ""))
    best_val = None
    if val:
        sc = np.array([v["summary"]["pq"] if v["summary"]["n_gt_instances"] > 0 else v["summary"]["wf1"] for v in val])
        best_i = int(np.nanargmax(sc))
        best_val, final = val[best_i]["summary"], val[-1]["summary"]
        if len(val) == 1:
            rows.append(row(INFO, "validation trend", "only one validation so far", ""))
        elif best_i == len(val) - 1:
            rows.append(row(WARN if len(ep) >= args["epochs"] else INFO, "validation trend",
                            f"best score {sc[best_i]:.3f} is the latest evaluation",
                            "still improving: raise --epochs or resume" if len(ep) >= args["epochs"] else ""))
        elif sc[-1] < sc[best_i] - 0.03:
            rows.append(row(WARN, "validation trend", f"best {sc[best_i]:.3f} at epoch {val[best_i]['epoch']}, latest {sc[-1]:.3f}",
                            "val fell after its peak: more augmentation, --dropout 0.1, or a smaller --lr-step"))
        else:
            rows.append(row(PASS, "validation trend", f"best {sc[best_i]:.3f} at epoch {val[best_i]['epoch']}, latest {sc[-1]:.3f}", ""))
        gap = final["loss"] / max(last["loss"], 1e-9)
        rows.append(row(WARN if gap > 1.6 else PASS, "train/val loss gap",
                        f"val loss {final['loss']:.3f} vs train {last['loss']:.3f} (ratio {gap:.2f}; train is augmented, val is not)",
                        "overfitting: more data, --dropout 0.1, stronger augmentation" if gap > 1.6 else ""))
        weak = [c for c in TARGET_CLASSES if final["support"][c] > 0 and final["f1_per_class"][c] < 0.5]
        rows.append(row(WARN if weak else PASS, "per-class F1", "below 0.5: " + (", ".join(f"{c} {final['f1_per_class'][c]:.2f}" for c in weak) or "none"),
                        "raise those class weights or add --focal-gamma 2; confirm the class exists in train" if weak else ""))
        rows.append(row(WARN if min(final["edge_precision"], final["edge_recall"]) < 0.6 else PASS, "Head 2",
                        f"precision {final['edge_precision']:.2f}, recall {final['edge_recall']:.2f} at threshold {final['thr']}",
                        "low recall splits instances; low precision merges neighbours: sweep the threshold, then adjust --lam"
                        if min(final["edge_precision"], final["edge_recall"]) < 0.6 else ""))
        sw = pd.DataFrame(final["sweep"])
        if sw["pq"].notna().any():
            top = sw.loc[sw["pq"].idxmax()]
            cur = sw.loc[(sw["thr"] - final["thr"]).abs().idxmin()]
            rows.append(row(INFO, "threshold sweep", f"best PQ {top['pq']:.3f} at {top['thr']}, current {cur['pq']:.3f} at {cur['thr']}",
                            f"use threshold {top['thr']}" if top["pq"] - cur["pq"] > 0.01 else ""))
        conf = np.array(final["conf"])
        off = conf - np.diag(np.diag(conf))
        flat = np.argsort(off.ravel())[::-1][:3]
        rows.append(row(INFO, "top confusions", "; ".join(f"{TARGET_CLASSES[i // NUM_CLASSES]} read as {TARGET_CLASSES[i % NUM_CLASSES]} ({int(off.ravel()[i]):,})"
                                                        for i in flat if off.ravel()[i] > 0) or "none", ""))
        rows.append(row(INFO, "reference", f"length-weighted F1 {final['wf1']:.3f} vs GAT-CADNet 0.850 (published, full FloorPlanCAD, different class set; not directly comparable)", ""))
    else:
        rows.append(row(WARN, "validation", "no validation entry yet", "lower --eval-every"))
    rows.append(row(INFO, "speed", f"{last['tiles_per_s']:.1f} tiles/s, {last['seconds']:.0f} s per epoch, peak GPU {last['gpu_peak_gb']:.1f} GB", ""))
    return show(rows), best_val


def plot_run(run_dir):
    run = Path(run_dir)
    log, steps = read_jsonl(run / "metrics.jsonl"), read_jsonl(run / "steps.jsonl")
    args = json.loads((run / "run_config.json").read_text())["args"]
    val = [r for r in log if r["event"] == "val" and r["set"] == args["val_set"]]
    fig, ax = plt.subplots(2, 2, figsize=(12, 7))
    st = pd.DataFrame(steps)
    if len(st):
        ax[0, 0].plot(st["step"], st["node"], color=SERIES[0], label="node CE")
        ax[0, 0].plot(st["step"], st["edge"], color=SERIES[1], label="edge BCE")
        ax[0, 0].set_yscale("log")
        ax[0, 1].plot(st["step"], st["grad_norm"], color=SERIES[2], label="grad norm")
        ax[0, 1].axhline(args["grad_clip"], color="#777", lw=1, ls="--", label="clip")
        ax[0, 1].set_yscale("log")
        ax[1, 1].plot(st["step"], st["lr"], color=SERIES[3], label="lr")
    if val:
        e = [v["epoch"] + 1 for v in val]
        for key, c in (("wf1", SERIES[0]), ("f1_macro", SERIES[1]), ("pq", SERIES[2]), ("edge_f1", SERIES[3])):
            ax[1, 0].plot(e, [v["summary"][key] for v in val], marker="o", color=c, label=key)
        ax[1, 0].set_ylim(0, 1)
    titles = [("training loss", "step"), ("gradient norm", "step"), ("validation", "epoch"), ("learning rate", "step")]
    for a, (t, xl) in zip(ax.ravel(), titles):
        a.set_title(t, loc="left", fontsize=11)
        a.set_xlabel(xl)
        a.grid(alpha=0.2)
        for s in ("top", "right"):
            a.spines[s].set_visible(False)
        if a.get_legend_handles_labels()[0]:
            a.legend(frameon=False)
    fig.tight_layout()
    return fig


def draw_tile(ax, tile, labels, title):
    pts = tile["pts"].astype(np.float64)
    segs = np.concatenate([np.stack([pts[:, k], pts[:, k + 1]], 1) for k in range(3)])
    cmap = plt.get_cmap("tab20")
    cols = np.array([(0.85, 0.85, 0.85, 1.0) if c == TARGET_CLASSES.index("other") else cmap(c) for c in range(NUM_CLASSES)])
    ax.add_collection(LineCollection(segs, colors=np.tile(cols[labels], (3, 1)), linewidths=0.9))
    ax.set_xlim(pts[..., 0].min(), pts[..., 0].max())
    ax.set_ylim(pts[..., 1].min(), pts[..., 1].max())
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title, loc="left", fontsize=10)


@torch.no_grad()
def show_prediction(model, tile, device, title="validation tile"):
    model.eval()
    b = to_device(collate([tile]), device)
    pred = model(b["x"], b["edge_index"], b["edge_attr"])["node_logits"].argmax(1).cpu().numpy()
    fig, ax = plt.subplots(1, 2, figsize=(13, 6.5))
    draw_tile(ax[0], tile, tile["y"].astype(int), f"{title}: ground truth")
    acc = float((pred == tile["y"]).mean())
    draw_tile(ax[1], tile, pred, f"prediction (primitive accuracy {acc:.2f})")
    cmap = plt.get_cmap("tab20")
    handles = [plt.Line2D([0], [0], color=(0.85, 0.85, 0.85) if c == "other" else cmap(i), lw=3, label=c) for i, c in enumerate(TARGET_CLASSES)]
    fig.legend(handles=handles, loc="lower center", ncol=6, frameon=False)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    return fig

### 2f. `gnnsynth.py`: a synthetic dataset for the smoke test
Draws simple floor plans (double-line walls, door arcs with leaves, three-line windows, stair treads, a bed) as FloorPlanCAD-style SVG and runs them through the real `fpgraph` tiler, so the shards have exactly the Phase A format. It is only used when `SMOKE_TEST = True`.

In [ ]:
%%writefile gnnsynth.py
import math
from pathlib import Path

import numpy as np

import fpgraph

SEM = {"wall": 1, "single door": 3, "double door": 4, "sliding door": 5, "window": 9, "stairs": 30, "bed": 14}


def _path(d, sem, inst=-1):
    return f'<path d="{d}" fill="none" stroke="#000" stroke-width="0.1" semantic-id="{sem}" instance-id="{inst}"/>'


def _wall(x0, y0, x1, y1, t=1.0):
    horiz = abs(y1 - y0) < abs(x1 - x0)
    o = (0, t) if horiz else (t, 0)
    return [_path(f"M {x0 - o[0]:.3f} {y0 - o[1]:.3f} L {x1 - o[0]:.3f} {y1 - o[1]:.3f}", SEM["wall"]),
            _path(f"M {x0 + o[0]:.3f} {y0 + o[1]:.3f} L {x1 + o[0]:.3f} {y1 + o[1]:.3f}", SEM["wall"])]


def _door(x, y, horiz, inst, double, r=8.0):
    if horiz:
        pts = [(x, y, x + r, y + r, x + r, y)] if not double else [(x, y, x + r, y + r, x + r, y), (x + 2 * r, y, x + r, y + r, x + r, y)]
        out = []
        for ax, ay, bx, by, cx, cy in pts:
            out.append(_path(f"M {ax:.3f} {ay:.3f} A {r} {r} 0 0 1 {bx:.3f} {by:.3f}", SEM["double door" if double else "single door"], inst))
            out.append(_path(f"M {ax:.3f} {ay:.3f} L {ax:.3f} {ay + r:.3f}", SEM["double door" if double else "single door"], inst))
        return out
    return [_path(f"M {x:.3f} {y:.3f} A {r} {r} 0 0 0 {x + r:.3f} {y + r:.3f}", SEM["single door"], inst),
            _path(f"M {x:.3f} {y:.3f} L {x + r:.3f} {y:.3f}", SEM["single door"], inst)]


def _window(x, y, horiz, inst, w=12.0):
    out = []
    for off in (-0.6, 0.0, 0.6):
        d = f"M {x:.3f} {y + off:.3f} L {x + w:.3f} {y + off:.3f}" if horiz else f"M {x + off:.3f} {y:.3f} L {x + off:.3f} {y + w:.3f}"
        out.append(_path(d, SEM["window"], inst))
    return out


def _stairs(x, y, inst, n=9, w=10.0, step=2.5):
    out = [_path(f"M {x:.3f} {y + i * step:.3f} L {x + w:.3f} {y + i * step:.3f}", SEM["stairs"], inst) for i in range(n)]
    out.append(_path(f"M {x:.3f} {y:.3f} L {x:.3f} {y + (n - 1) * step:.3f}", SEM["stairs"], inst))
    out.append(_path(f"M {x + w:.3f} {y:.3f} L {x + w:.3f} {y + (n - 1) * step:.3f}", SEM["stairs"], inst))
    return out


def synth_svg(rng):
    cut_x = float(rng.uniform(35, 65))
    cut_y = float(rng.uniform(35, 65))
    els, inst = [], 0
    for seg in [(5, 5, 95, 5), (5, 95, 95, 95), (5, 5, 5, 95), (95, 5, 95, 95), (cut_x, 5, cut_x, 95), (5, cut_y, 95, cut_y)]:
        els += _wall(*seg)
    for _ in range(int(rng.integers(3, 6))):
        horiz = bool(rng.integers(2))
        if horiz:
            x, y = float(rng.uniform(8, 80)), cut_y
        else:
            x, y = cut_x, float(rng.uniform(8, 80))
        els += _door(x, y, horiz, inst, double=bool(rng.random() < 0.3)) if horiz else _door(x, y, False, inst, False)
        inst += 1
    for _ in range(int(rng.integers(2, 5))):
        side = int(rng.integers(4))
        t = float(rng.uniform(10, 75))
        x, y, horiz = [(t, 5, True), (t, 95, True), (5, t, False), (95, t, False)][side]
        els += _window(x, y, horiz, inst)
        inst += 1
    els += _stairs(float(rng.uniform(10, 25)), float(rng.uniform(10, 25)), inst)
    inst += 1
    bx, by = float(rng.uniform(55, 75)), float(rng.uniform(55, 75))
    els.append(_path(f"M {bx} {by} L {bx + 10} {by} L {bx + 10} {by + 7} L {bx} {by + 7} Z", SEM["bed"]))
    return ('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 100 100">' + "".join(els) + "</svg>")


def build_synthetic(out_dir, cfg_json, n_train=24, n_val=6, n_test=6, seed=0):
    import json
    out_dir = Path(out_dir)
    svg_dir = out_dir / "_svg"
    svg_dir.mkdir(parents=True, exist_ok=True)
    conf = json.loads(Path(cfg_json).read_text())
    cfg = fpgraph.GraphConfig(**conf["graph"])
    lut = fpgraph.build_label_map(conf["raw_class_names"], conf["harmonization"], conf["targets"])
    stuff = fpgraph.stuff_mask(conf["targets"])
    rng = np.random.default_rng(seed)
    for split, n in (("train", n_train), ("val", n_val), ("test", n_test)):
        paths = []
        for i in range(n):
            p = svg_dir / f"{split}_{i:03d}.svg"
            p.write_text(synth_svg(rng))
            paths.append(p)
        fpgraph.process_split(paths, split, out_dir, cfg, lut, stuff, workers=1, tiles_per_shard=16, log_every=10 ** 9)
    return out_dir

## 3. Load the data
With `SMOKE_TEST = False` this downloads, from `HF_DATASET_REPO`, the FloorPlanCAD `train` and `val` shards (`graphs/train`, `graphs/val`) and, if `DOWNLOAD_INDIAN`, the Phase B shards under `indian/graphs/{train,val}`. Test shards are fetched only in step 8. `MAX_FP_TRAIN_SHARDS` and `MAX_FP_VAL_SHARDS` cap the download for a quick trial (0 means all; a shard holds up to 512 tiles).

In [ ]:
if SMOKE_TEST:
    import gnnsynth
    gnnsynth.build_synthetic(DATA_DIR, "config.json", seed=0)
    gnnsynth.build_synthetic(DATA_DIR / "indian", "config.json", n_train=8, n_val=3, n_test=3, seed=1)
else:
    api = HfApi(token=HF_TOKEN)
    repo_files = api.list_repo_files(HF_DATASET_REPO, repo_type="dataset")

    def shard_files(prefix, split, limit):
        found = sorted(f for f in repo_files if f.startswith(f"{prefix}graphs/{split}/") and f.endswith(".npz"))
        return found[:limit] if limit else found

    wanted = shard_files("", "train", MAX_FP_TRAIN_SHARDS) + shard_files("", "val", MAX_FP_VAL_SHARDS)
    if DOWNLOAD_INDIAN:
        wanted += shard_files("indian/", "train", 0) + shard_files("indian/", "val", 0)
    print(len(wanted), "shard files to fetch")
    snapshot_download(HF_DATASET_REPO, repo_type="dataset", allow_patterns=wanted, local_dir=str(DATA_DIR), token=HF_TOKEN)

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from gnnmodel import FloorGAT, ModelConfig, PanopticLoss
from gnndata import load_tiles, class_stats
from gnncheck import (check_data, check_augmentation, check_model, overfit_test, lr_sweep, dry_run, project_time,
                      analyze_run, plot_run, show_prediction, show)
from gnneval import evaluate_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NGPU = max(torch.cuda.device_count(), 1)
print("device:", device, "| gpus:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

fp_train = load_tiles(DATA_DIR, "train", 0)
fp_val = load_tiles(DATA_DIR, "val", 0)
HAS_INDIAN = (DATA_DIR / "indian" / "graphs" / "train").exists()
ind_train = load_tiles(DATA_DIR / "indian", "train", 1) if HAS_INDIAN else []
ind_val = load_tiles(DATA_DIR / "indian", "val", 1) if HAS_INDIAN else []
print(f"FloorPlanCAD tiles: train {len(fp_train)}, val {len(fp_val)} | Indian tiles: train {len(ind_train)}, val {len(ind_val)}")

fp_stats = class_stats(fp_train)
loss_fn = PanopticLoss(fp_stats["weights"], fp_stats["pos_weight"]).to(device)
MODEL_CFG = ModelConfig()
pd.DataFrame({"class": PHASE_A["targets"], "train nodes": fp_stats["counts"], "weight": fp_stats["weights"].round(3)})

## 4. Verification A: is the setup correct, before training?
Run these before any long run. They take a few minutes and cost almost nothing.

### 4.1 The data
Tile invariants (indices in range, `i < j`, no degree above 30, finite features, labels in range, positive edges only inside the mask), class balance, and which classes are absent. An absent class has weight 0 and cannot be learned, so for the Indian set expect `curtain_wall`, `door_other`, `opening` and `elevator` to show here.

In [ ]:
PRE = []
rows = check_data(fp_train, "FloorPlanCAD train") + check_data(fp_val, "FloorPlanCAD val")
if HAS_INDIAN:
    rows += check_data(ind_train, "Indian train")
PRE += rows
_ = show(rows)

### 4.2 The augmentation
The test that matters: applying "rotate by 0, no mirror" through the recompute path must give back the features Phase A stored. If it does not, the augmentation formulas are different from the ones the data was built with.

In [ ]:
rows = check_augmentation(fp_train)
PRE += rows
_ = show(rows)

### 4.3 The model
- Initial loss. With near-zero head weights the node loss must be about `ln 12 = 2.485`. The edge loss is predictable from the positive fraction and `pos_weight`.
- Equivariance. Shuffle the node order: outputs must shuffle the same way, to numerical precision.
- Batch independence. A tile's output must be identical alone or inside a batch.
- Gradient coverage. After one backward pass every parameter has a finite, non-zero gradient.

In [ ]:
rows = check_model(MODEL_CFG, fp_train, loss_fn, device)
PRE += rows
_ = show(rows)

### 4.4 Overfit four tiles
The most informative single test. Take 4 tiles with many instances, train with no augmentation and a fixed learning rate. A correct model reaches almost 100% node accuracy and a high edge F1 within a few hundred steps.
- Passes: the model, loss, labels and features are wired correctly, and any later poor result is about data size or tuning.
- Fails: do **not** launch a long run. Read the "what to change" column, and look at whether the labels in those tiles match what you see in step 6's picture.

In [ ]:
hist, rows = overfit_test(MODEL_CFG, fp_train, loss_fn, device, steps=300)
PRE += rows
_ = show(rows)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist["step"], hist["loss"], color="#2a6fdb")
ax[0].set_yscale("log")
ax[0].set_title("loss", loc="left")
ax[1].plot(hist["step"], hist["node_acc"], color="#1a9850", label="node accuracy")
ax[1].plot(hist["step"], hist["edge_f1"], color="#e07b00", label="edge F1")
ax[1].set_ylim(0, 1.02)
ax[1].legend(frameon=False)
ax[1].set_title("overfit progress", loc="left")
for a in ax:
    a.set_xlabel("step")
    a.grid(alpha=0.2)
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
plt.tight_layout()

### 4.5 Learning-rate sweep
80 steps at each of four learning rates on the same small batches. This does not choose the final rate (80 steps is short), but it catches two things: a rate that diverges, and a rate that is clearly slower. The plan's starting point is `1e-3` (the published reference value).

In [ ]:
sweep_df, BEST_LR = lr_sweep(MODEL_CFG, fp_train, loss_fn, device)
display(sweep_df)
print("lowest final loss at lr =", BEST_LR, "| configured LR =", LR)
if not sweep_df["finite"].all():
    print("WARNING: a learning rate diverged:", sweep_df.loc[~sweep_df["finite"], "lr"].tolist())

### 4.6 Go / no-go

In [ ]:
pre = pd.DataFrame(PRE)
fails, warns = pre[pre["status"] == "FAIL"], pre[pre["status"] == "WARN"]
print(f"{len(pre)} checks: {(pre['status'] == 'PASS').sum()} pass, {len(warns)} warn, {len(fails)} fail")
if len(fails):
    print("NO-GO. Fix these first:")
    display(fails)
elif len(warns):
    print("GO WITH CARE. Read these:")
    display(warns)
else:
    print("GO.")

## 5. D1 dry run: measure before committing hours
Runs a few training steps on the largest tile alone and on batches at three node budgets, and records **measured** peak GPU memory and seconds per step. Then it projects the pre-training time from the measured speed (a projection, not a measurement: it assumes DDP scales linearly over `NGPU` GPUs and ignores validation).

- Pick the node budget with the largest batch whose peak memory is comfortably under your GPU's memory (a T4 has 16 GB). The configured `NODE_BUDGET` is used by training.
- The plan's estimate was about 3-6 h for 100 epochs. Compare.

In [ ]:
budgets = (NODE_BUDGET // 2, NODE_BUDGET, NODE_BUDGET * 2)
dry = dry_run(fp_train, MODEL_CFG, loss_fn, device, budgets=budgets, max_tiles=MAX_TILES, amp=USE_AMP)
display(dry)
proj = project_time(dry, f"budget {NODE_BUDGET}", len(fp_train), PRETRAIN_EPOCHS, NGPU)
print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in proj.items()})

## 6. D2 pre-train on FloorPlanCAD
`launch` starts `train_gnn.py` (one process, or `torch.distributed.run` with one process per GPU) and streams its output. Running the same cell again **resumes** from `last.pt`.

While it trains you can run step 6.1 below from a second notebook cell or after a disconnect, because it only reads the log files.

In [ ]:
WORKERS = 0 if SMOKE_TEST else 2
SMOKE_ARGS = ["--eval-every", "1", "--warmup-steps", "5", "--patience", "3"] if SMOKE_TEST else []


def launch(args, nproc=1):
    if nproc > 1:
        cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={nproc}", "train_gnn.py"] + args
    else:
        cmd = [sys.executable, "train_gnn.py"] + args
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    print("exit code", proc.returncode)
    return proc.returncode


def common_args(out):
    args = ["--fp-dir", str(DATA_DIR), "--ind-dir", str(DATA_DIR / "indian"), "--out", str(out),
            "--node-budget", str(NODE_BUDGET), "--max-tiles", str(MAX_TILES), "--workers", str(WORKERS)]
    args += ["--amp"] if USE_AMP else []
    args += ["--wandb", WANDB_PROJECT] if WANDB_PROJECT else []
    return args + SMOKE_ARGS


PRETRAIN_DIR = RUN_DIR / "pretrain"
launch(common_args(PRETRAIN_DIR) + ["--mode", "pretrain", "--epochs", str(PRETRAIN_EPOCHS), "--lr", str(LR),
                                    "--val-set", "fp"], NGPU)

### 6.1 Verification B: was the training correct?
`analyze_run` reads `run_config.json`, `steps.jsonl` and `metrics.jsonl` and applies these rules, each printed with a PASS / WARN / FAIL and a suggested change:

| Finding | Rule | Usual fix |
|---|---|---|
| numerics | any non-finite loss, or more than 0.5% of steps skipped | lower `--lr`, drop `--amp` |
| learning | train loss should fall by at least 30% across the run | check the overfit test; try another lr |
| gradient spikes | median ratio of max to mean gradient norm above 10 | lower `--lr` |
| clipping | mean norm above 3x the clip in most epochs | lower `--lr` |
| validation trend | best score is the latest (still improving) or has fallen after its peak | train longer / more augmentation, dropout |
| train/val gap | val loss more than 1.6x train loss (train is augmented, val is not, so some gap is normal) | more data, dropout |
| per-class F1 | any class present in val with F1 under 0.5 | class weight up, `--focal-gamma 2`, check the class exists in train |
| Head 2 | edge precision or recall under 0.6 | low recall splits instances, low precision merges neighbours: sweep threshold, then `--lam` |
| threshold sweep | best PQ threshold differs from the current by more than 0.01 PQ | use that threshold |
| top confusions | the three largest off-diagonal pairs | informational |

The curves show the same information. In a healthy run the node loss falls steadily, the gradient norm settles under the clip line, and validation F1 and PQ rise and then flatten.

In [ ]:
diag_pre, best_pre = analyze_run(PRETRAIN_DIR)
plot_run(PRETRAIN_DIR)
plt.show()

### 6.2 Look at a prediction
Numbers can hide a systematic mistake (for example every door arc read as a window). This draws one validation tile twice, ground truth on the left and the model on the right, in the same colours. Check that doors are doors, walls are walls, and that the errors are plausible.

In [ ]:
ck = torch.load(PRETRAIN_DIR / "best.pt", map_location=device, weights_only=False)
model = FloorGAT(ModelConfig(**ck["cfg"])).to(device)
model.load_state_dict(ck["model"])
tile = max(fp_val, key=lambda t: int(t["inst"].max() + 1) if len(t["inst"]) else 0)
show_prediction(model, tile, device, "FloorPlanCAD val")
plt.show()

## 7. D3 fine-tune on the Indian shards
Starts from the best pre-trained checkpoint with a lower learning rate (`1e-4`), uses all Indian train tiles plus FloorPlanCAD tiles for about 25% of each epoch (so the model does not forget), blends the class weights of the two sources, and freezes the first three GAT stages (zero learning rate) for the first 4 epochs. Checkpoint selection is on **Indian val** PQ, and FloorPlanCAD val is logged alongside as a forgetting check.

Remember what the Indian split is: all floors come from one office, so Indian val and test are **same-drafter**. A high Indian number shows the model has adapted to this drafting style; it does not show it generalises to other offices. Skip this step if `HAS_INDIAN` is false.

In [ ]:
FINETUNE_DIR = RUN_DIR / "finetune"
if not HAS_INDIAN:
    print("no Indian shards loaded; skipping fine-tune")
else:
    freeze_epochs = 1 if SMOKE_TEST else 4
    launch(common_args(FINETUNE_DIR) + ["--mode", "finetune", "--init", str(PRETRAIN_DIR / "best.pt"),
                                        "--epochs", str(FINETUNE_EPOCHS), "--lr", str(FINETUNE_LR),
                                        "--freeze-stages", "3", "--freeze-epochs", str(freeze_epochs),
                                        "--fp-mix", "0.25", "--val-set", "ind", "--aux-val", "fp",
                                        "--lr-step", "20", "--warmup-steps", "50"], NGPU)

### 7.1 Verification B for the fine-tune
Same diagnosis on the Indian validation set. Extra things to read:
- The Indian train set has no elevator, curtain wall, door_other or opening tiles, so those classes show F1 of 0 or no support. That is the data, not a bug.
- Compare FloorPlanCAD val F1 against the pre-training value: a large drop means the 25% mix is too small or the learning rate too high.
- The Indian val set is small (tens of tiles), so PQ moves in big steps between epochs; judge the trend, not single epochs.

In [ ]:
if HAS_INDIAN:
    diag_ft, best_ft = analyze_run(FINETUNE_DIR)
    plot_run(FINETUNE_DIR)
    plt.show()
    aux = [json.loads(l) for l in (FINETUNE_DIR / "metrics.jsonl").read_text().splitlines()]
    aux = [r for r in aux if r["event"] == "val" and r["set"] == "fp"]
    if aux:
        print("FloorPlanCAD val during fine-tune: wF1 by epoch ->", [round(r["summary"]["wf1"], 3) for r in aux])

## 8. D4 Head 2 threshold and final metrics
**Threshold.** Head 2 gives a probability per edge, and the threshold decides which edges join primitives into one instance. The plan starts at 0.7. This sweeps 0.30 to 0.90 on the Indian val set, picks the threshold with the best PQ (ties go to the one closest to 0.7), and saves it beside the checkpoint as `head2_threshold.json`.

**Final metrics.** Both checkpoints are scored on both test sets: pre-trained and fine-tuned, on FloorPlanCAD test (a downloaded subset of `MAX_FP_TEST_SHARDS` shards; 0 means all) and on Indian test (TOWER-B ground floor, the floor held out for the classical-versus-GNN comparison). The fine-tuned model should gain on the Indian test and lose little on FloorPlanCAD.

In [ ]:
THRESHOLDS = [round(t, 2) for t in np.arange(0.3, 0.91, 0.05)]
FINAL_CKPT = FINETUNE_DIR / "best.pt" if HAS_INDIAN else PRETRAIN_DIR / "best.pt"
FINAL_CFG = FINETUNE_DIR / "run_config.json" if HAS_INDIAN else PRETRAIN_DIR / "run_config.json"
sel_root, sel_src = (DATA_DIR / "indian", 1) if HAS_INDIAN else (DATA_DIR, 0)

ev, n_tiles = evaluate_checkpoint(FINAL_CKPT, FINAL_CFG, sel_root, "val", sel_src, THRESHOLDS, device, NODE_BUDGET, MAX_TILES, USE_AMP)
sweep = pd.DataFrame(ev.summarize(0.7)["sweep"])
display(sweep)
valid = sweep.dropna(subset=["pq"])
if len(valid):
    top = valid[valid["pq"] == valid["pq"].max()]
    HEAD2_THR = float(top.iloc[(top["thr"] - 0.7).abs().argsort()].iloc[0]["thr"])
else:
    HEAD2_THR = 0.7
(FINAL_CKPT.parent / "head2_threshold.json").write_text(json.dumps({"threshold": HEAD2_THR, "selected_on": f"{'indian' if HAS_INDIAN else 'fp'} val, {n_tiles} tiles",
                                                                   "sweep": sweep.to_dict("records")}, indent=1))
print("chosen Head 2 threshold:", HEAD2_THR)

In [ ]:
if not SMOKE_TEST:
    require("HF_DATASET_REPO", HF_DATASET_REPO)
    test_files = sorted(f for f in HfApi(token=HF_TOKEN).list_repo_files(HF_DATASET_REPO, repo_type="dataset")
                        if f.startswith("graphs/test/") and f.endswith(".npz"))
    test_files = test_files[:MAX_FP_TEST_SHARDS] if MAX_FP_TEST_SHARDS else test_files
    ind_test = [f for f in HfApi(token=HF_TOKEN).list_repo_files(HF_DATASET_REPO, repo_type="dataset")
                if f.startswith("indian/graphs/test/") and f.endswith(".npz")] if HAS_INDIAN else []
    snapshot_download(HF_DATASET_REPO, repo_type="dataset", allow_patterns=test_files + ind_test, local_dir=str(DATA_DIR), token=HF_TOKEN)

models = {"pretrained": (PRETRAIN_DIR / "best.pt", PRETRAIN_DIR / "run_config.json")}
if HAS_INDIAN:
    models["fine-tuned"] = (FINETUNE_DIR / "best.pt", FINETUNE_DIR / "run_config.json")
tests = {"FloorPlanCAD test": (DATA_DIR, 0)}
if HAS_INDIAN and (DATA_DIR / "indian" / "graphs" / "test").exists():
    tests["Indian test (same drafter)"] = (DATA_DIR / "indian", 1)

RESULTS, table = {}, []
for mname, (ck_path, cfg_path) in models.items():
    for tname, (root, src) in tests.items():
        ev, n = evaluate_checkpoint(ck_path, cfg_path, root, "test", src, THRESHOLDS, device, NODE_BUDGET, MAX_TILES, USE_AMP)
        s = ev.summarize(HEAD2_THR)
        RESULTS[f"{mname} | {tname}"] = {k: v for k, v in s.items() if k not in ("conf", "sweep")} | {"tiles": n}
        table.append({"model": mname, "test set": tname, "tiles": n, "wF1": s["wf1"], "macro F1": s["f1_macro"],
                      "PQ": s["pq"], "RQ": s["rq"], "SQ": s["sq"], "edge F1": s["edge_f1"]})
results_df = pd.DataFrame(table).round(3)
display(results_df)
Path("results.json").write_text(json.dumps({"head2_threshold": HEAD2_THR, "results": RESULTS}, indent=1, default=float))
display(pd.DataFrame({k: v["f1_per_class"] for k, v in RESULTS.items()}).round(3))

## 9. Push the trained model to Hugging Face
Uploads to `HF_OUTPUT_REPO` (a **model** repo, created if missing, private by default through `HF_OUTPUT_PRIVATE`). The token needs write access.

What goes up, and why:
- `best.pt` for pre-train and fine-tune: the selected weights, with the model config and validation summary inside. `last.pt` (weights plus optimiser, several times larger) is only uploaded if `PUSH_LAST_CHECKPOINT = True`.
- `run_config.json`: arguments, parameter count, tile counts, class weights and edge `pos_weight`. Needed to rebuild the loss and to reproduce the run.
- `metrics.jsonl`, `steps.jsonl`: the training logs behind the diagnosis.
- `head2_threshold.json` and `results.json`: the chosen threshold and the final test metrics.
- The five `.py` modules and `config.json`: the code needed to load the checkpoint in Phase E (`FloorGAT(ModelConfig(**ckpt["cfg"]))`).
- A short `README.md` model card, with the caveats stated.

Nothing from the dataset itself is uploaded here, only derived model artefacts. Skipped when `SMOKE_TEST` is true.

In [ ]:
if SMOKE_TEST:
    print("smoke test: nothing pushed")
else:
    require("HF_OUTPUT_REPO", HF_OUTPUT_REPO)
    stage = Path("hub_upload")
    shutil.rmtree(stage, ignore_errors=True)
    stage.mkdir()
    names = ["best.pt", "run_config.json", "metrics.jsonl", "steps.jsonl", "head2_threshold.json"]
    names += ["last.pt"] if PUSH_LAST_CHECKPOINT else []
    for run in (PRETRAIN_DIR, FINETUNE_DIR):
        for n in names:
            if (run / n).exists():
                (stage / run.name).mkdir(exist_ok=True)
                shutil.copy(run / n, stage / run.name / n)
    for n in ["gnnmodel.py", "gnndata.py", "gnneval.py", "train_gnn.py", "gnncheck.py", "config.json", "results.json"]:
        shutil.copy(n, stage / n)
    card = ["# Floor-plan symbol-spotting GNN (CV5570, Team Visionaries)", "",
            "Sparse GAT, 8 stages x 8 heads, two heads: 12-class primitive semantics and same-instance edge classification.",
            "Pre-trained on FloorPlanCAD graph tiles, fine-tuned on auto-labelled Indian drawings.", "",
            f"Head 2 threshold: {HEAD2_THR}", "",
            "## Test results", "", "```", results_df.to_string(index=False), "```", "",
            "## Caveats",
            "- Indian labels are automatic (silver) and were not hand-verified.",
            "- Indian val and test floors come from one office (same drafter); they do not measure generalisation to other offices.",
            "- Indian train has no elevator, curtain wall, door_other or opening examples.",
            "- Pre-training used a 2,500-drawing FloorPlanCAD subset; scores are not comparable to published full-dataset results.", "",
            "Load: `ck = torch.load('finetune/best.pt', weights_only=False); model = FloorGAT(ModelConfig(**ck['cfg'])); model.load_state_dict(ck['model'])`"]
    (stage / "README.md").write_text("\n".join(card))
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_OUTPUT_REPO, repo_type="model", private=HF_OUTPUT_PRIVATE, exist_ok=True)
    api.upload_folder(folder_path=str(stage), repo_id=HF_OUTPUT_REPO, repo_type="model", commit_message="Phase C/D: pre-trained and fine-tuned GNN")
    print("uploaded to", HF_OUTPUT_REPO)